# <font color="#418FDE" size="6.5" uppercase>**A: Transformers Essentials**</font>
----

> Last update: 20241102

By the end of this lecture, you will be able to:

* Explain the Attention Mechanism & its components in transformer models.
* Identify Large Language Models (LLMs) tasks, architectures, data structures, & major models.  
* Develop prototype multi-head multi-layer transformers from scratch for Mechanical Engineering text generation.

## **1. Introduction to the Attention Mechanism in Machine Learning**

In the realm of natural language processing (NLP), the “Attention” mechanism has emerged as a revolutionary concept that has significantly enhanced the performance of machine learning models in tasks involving language understanding & neration. For mechanical engineering students familiar with traditional models like Long Short-Term Memory networks (LSTMs) & Support Vector Machines (SVMs), understanding the Attention mechanism provides a gateway to leveraging advanced models like Transformers & Large Language Models (LLMs) for complex engineering tasks.

At its essence, the Attention mechanism allows models to focus on specific parts of the input data when generating each part of the output. In the context of NLP, this means that when generating or analyzing a word in a sentence, the model can “attend” to other relevant words in the input, rather than processing the input sequentially or treating all input words equally.

For example, consider the sentence: “The turbine that was installed last year failed due to overheating.” When analyzing or generating text related to this sentence, the model needs to understand the relationships between “turbine,” “installed,” “failed,” & “overheating.” The Attention mechanism enables the model to weigh the importance of these words relative to each other, allowing for a more nuanced understanding.

### **1.1. Self-Attention Head in Encoder**

<div align="left">
  <img src="https://github.com/mhrafiei/figures/blob/main/535_743/module_12/self_attention_head_cliped.png?raw=true" width="150%">
  <br>
  <figcaption>Figure: Self-Attention Head Example</figcaption>
</div>

The figure represents the self-attention process in a Transformer model, specifically focusing on the steps involved in calculating context vectors for each token in a sequence.

> **1. Tokenization & Embeddings (Left Side of the figure)**
* On the left side of the figure, you have a list of tokens from a sentence: “The turbine that was installed last year failed due to overheating.”
* Each word (token) is tokenized into its corresponding token ID. For example:
    * The → Token ID: 504
    * turbine → Token ID: 122
    * that → Token ID: 748
    * …
* These token IDs are mapped to embedding vectors, which are typically learned during the model’s training. In this case, the embeddings for each token are represented as $\mathbf{E}_1, \mathbf{E}_2, \dots, \mathbf{E}_{11}$, where each $\mathbf{E}_i$ is the embedding of the $i^{th}$ token in the sequence.
Each embedding $\mathbf{E}_i$ contains positional information & token information, which are added together to capture both the token’s identity & its position in the sequence.

> **2. Query, Key, & Value Calculation (Blue Box)**
* Each token’s embedding $\mathbf{E}_i$ is then passed through three separate linear transformations to produce:
    * Queries $\mathbf{Q}_i$
    * Keys $\mathbf{K}_i$
    * Values $\mathbf{V}_i$
* These transformations are performed using learned weight matrices:
    * $\mathbf{Q}_i = \mathbf{W}_\text{query} \cdot \mathbf{E}_i$
    * $\mathbf{K}_i = \mathbf{W}_\text{key} \cdot \mathbf{E}_i$
    * $\mathbf{V}_i = \mathbf{W}_\text{value} \cdot \mathbf{E}_i$
* $\mathbf{W}_\text{query}$, $\mathbf{W}_\text{key}$, & $\mathbf{W}_\text{value}$ are learned weight matrices that map the embedding  $\mathbf{E}_i$ to the query, key, & value vectors, respectively.

> **3. Attention Score Calculation (Purple Box)**
* For each token $i$, the attention score  $\mathbf{S}_{i,j}$  between the token $i$ & every other token $j$ in the sequence is calculated by taking the dot product of the query vector $\mathbf{Q}_i$ & the key vector $\mathbf{K}_j$:
    * $\mathbf{S}_\text{i,j} = \frac{\mathbf{Q}_i \cdot {\mathbf{K}_j}^T}{\sqrt{D}}$
* Here, $D$ is the dimensionality of the key vectors, & the division by $\sqrt{D}$ is done to ensure that the dot product values do not grow too large.
* This produces a score $\mathbf{S}_\text{i,j}$ that indicates how much token $i$ should “attend to” token $j$.

> **4. Attention Weights Calculation (Green Box)**
* Once the attention scores $\mathbf{S}_\text{i,j}$ are computed, they are passed through a softmax function to produce the attention weights $\mathbf{A}_\text{i,j}$:
    * $\mathbf{A}_\text{i,j} = \text{softmax}(S_\text{i,j})$
* The softmax function normalizes the attention scores so that they sum to 1 across all tokens for each query token $i$ .
* These attention weights $\mathbf{A}_\text{i,j}$ determine how much importance token $i$ gives to token $j$ when generating its final representation (context vector).

> **5. Context Vector Calculation (Red Box)**
* The context vector $\mathbf{C}_i$ for each token $i$ is computed as a weighted sum of the value vectors $\mathbf{V}_j$, where the weights are the attention weights $\mathbf{A}_\text{i,j}$:
    * $\mathbf{C}_i = \sum_\text{j=1}^{I} (\mathbf{A}_\text{i,j} \cdot \mathbf{V}_j)$
* Each context vector $\mathbf{C}_i$ is a combination of all the other tokens’ value vectors, weighted by how much attention token i  gives to each other token $j$.
* Essentially, the context vector $\mathbf{C}_i$ captures a blend of the information from other tokens in the sequence, depending on how much attention the query token $i$  gave to them.

> **6. Final Context Vector Output**
* Once the context vectors $\mathbf{C}_1, \mathbf{C}_2, \dots, \mathbf{C}_{11}$ are computed for all tokens in the sequence, these context vectors are the final output of the self-attention head.
* These context vectors are then either passed to the next layer of the Transformer or used in further computations, such as cross-attention or generating final output probabilities (if we’re in the last layer of the decoder).




### **1.2. Encoder Architecture**





<div align="left">
  <img src="https://github.com/mhrafiei/figures/blob/main/535_743/module_12/encoder.png?raw=true" width="150%">
  <br>
  <figcaption>Figure: Encoder Architecture</figcaption>
</div>

The figure represents the **encoder architecture** for a single layer $l$ in a Transformer model. It illustrates how input embeddings are processed through self-attention heads, concatenation, residual connections, normalization, & feed-forward layers. For each token, we assume the embedding size is $1 \times 512$, & there are $N_l = 8$ attention heads.

> **1. Tokenization & Embeddings (Left Side)**
* The input sequence (e.g., a sentence) is **tokenized**, where each word or subword is mapped to a token (an integer ID).
* These tokens are converted into **embedding vectors** of size $1 \times 512$ using a learned embedding matrix, & **positional encodings** are added to capture the order of tokens in the sequence.
* For example, if the sentence is **"The turbine failed due to overheating"**, the token for **"turbine"** will be mapped to an embedding $\mathbf{E}_{\text{turbine}}$ of size $1 \times 512$. The entire input sentence will be represented as a sequence of embedding vectors for each token, each of size $1 \times 512$.

> **2. Splitting for Multi-Head Attention (Blue Box)**
* Each token’s embedding (size $1 \times 512$) is **split** into smaller parts to be processed by the $N_l = 8$ self-attention heads.
* Specifically, for each token embedding (e.g., $\mathbf{E}_{\text{turbine}}$ for **"turbine"**), we split the embedding into 8 smaller vectors:
    * $\mathbf{E}_\text{turbine, 1}, \mathbf{E}_\text{turbine, 2}, \dots, \mathbf{E}_\text{turbine, 8}$
    * Where each $\mathbf{E}_{\text{turbine}, n}$ has a size of $1 \times 64$.
* Each part is sent to a corresponding self-attention head for further processing.

> **3. Multi-Head Self-Attention (Dark Blue Box)**
* Each of the 8 **self-attention heads** processes its assigned part of the embedding independently, using its own **query**, **key**, & **value** transformations.
* For example, for the token **"turbine"**, the 8 attention heads work in parallel on their respective parts $\mathbf{E}_{\text{turbine}, 1}, \mathbf{E}_{\text{turbine}, 2}, \dots, \mathbf{E}_{\text{turbine}, 8}$, each computing its own attention scores & context vectors (as learned in the previous Colab text cell).
* The output of each attention head for **"turbine"** is a **context vector** of size $1 \times 64$.

> **4. Concatenation of Self-Attention Heads (Green Box)**
* After each of the 8 self-attention heads has computed its own **context vector** for the token **"turbine"** (each of size $1 \times 64$), the outputs are **concatenated** to form a single vector of size $1 \times 512$, which matches the original embedding size.
* This process ensures that the outputs from all attention heads are combined, providing a richer representation of the token **"turbine"** based on information from all the heads.

> **5. Residual Connection & Normalization (Purple Box)**
* A **residual connection** is applied by adding the original input embedding for the token to the concatenated output of the self-attention heads:
    * For example, for the token **"turbine"**, the residual connection results in:
        * $\mathbf{E}_\text{residual} = \mathbf{E}_\text{turbine} + \mathbf{C}_\text{concat}$
    * Here, $\mathbf{C}_\text{concat}$ is the concatenated output from the 8 self-attention heads (size $1 \times 512$), & $\mathbf{E}_\textbf{turbine}$ is the original embedding for the token (also size $1 \times 512$).
* After the residual connection, **layer normalization** is applied to the result to ensure stable training by normalizing the values & preventing gradient issues.

> **6. Fully Connected Layer (Orange Box)**
* The output of the self-attention mechanism is then passed through a **fully connected feed-forward network (FFN)**, which consists of two linear transformations with a non-linearity (e.g., ReLU) in between:
    * First, a linear transformation maps the normalized vector to a higher-dimensional space:
        * $\mathbf{Z}_1 = \mathbf{W}_1 \cdot \mathbf{E}_\text{normalized} + \mathbf{b}_1$
    * Next, an activation function (e.g., ReLU) is applied:
        * $\mathbf{Z}_2 = \text{ReLU}(\mathbf{Z}_1)$
    * Finally, another linear transformation maps the output back to the original dimensionality (size $1 \times 512$):
        * $\mathbf{E}_\text{FFN} = \mathbf{W}_2 \cdot \mathbf{Z}_2 + \mathbf{b}_2$

> **7. Residual Connection & Normalization (Second Residual Path)**
* After the output of the FFN, another **residual connection** is applied by adding the input to the FFN to the output of the FFN:
    * $\mathbf{E}_\text{residual, FFN} = \mathbf{E}_\text{normalized} + \mathbf{E}_\text{FFN}$
* A second **layer normalization** is then applied to this result.

> **8. Output of Layer $l$**
* The final output of layer $l$ is a normalized embedding of size $1 \times 512$ for each token, which can be passed to the next encoder layer.


### **1.3. Masked Self-Attention Head in Decoder**

<div align="left">
  <img src="https://github.com/mhrafiei/figures/blob/main/535_743/module_12/masked_self_attention_head.png?raw=true" width="150%">
  <br>
  <figcaption>Figure: Masked Self-Attention Head Example</figcaption>
</div>

The figure represents the masked self-attention head process in an autoregressive manner, focusing on how tokens are processed one-by-one during text generation. It is assumed that the decoder is to generate 256 tokens.

> **1. Autoregressive Embeddings (Left Side of the Figure)**
* The sequence of tokens being processed in the decoder starts with the **\[Start of Sequence (SOS)\]** token, followed by other tokens that are either generated or input during decoding.
* The figure shows the embeddings for tokens such as:
    * $\mathbf{T_1}$ (SOS), $\mathbf{T_2}$, ..., $\mathbf{T_k}$, ..., $\mathbf{T_{255}}$.
* Each token is tokenized into corresponding embeddings $\mathbf{E}_1, \mathbf{E}_2, \dots, \mathbf{E}_{255}$. These embeddings combine both token & positional information, just as in the encoder.

> **2. Query, Key, & Value Calculation (Blue Box)**
* Just like in the standard self-attention mechanism, each token’s embedding $\mathbf{E}_k$ is transformed into:
    * Query vector $\mathbf{Q}_k$
    * Key vector $\mathbf{K}_k$
    * Value vector $\mathbf{V}_k$
* These transformations are achieved using learned weight matrices:
    * $\mathbf{Q}_k = \mathbf{W}_\text{query} \cdot \mathbf{E}_k$
    * $\mathbf{K}_k = \mathbf{W}_\text{key} \cdot \mathbf{E}_k$
    * $\mathbf{V}_k = \mathbf{W}_\text{value} \cdot \mathbf{E}_k$
* $\mathbf{W}_\text{query}$, $\mathbf{W}_\text{key}$, & $\mathbf{W}_\text{value}$ are learned matrices that transform the embeddings $\mathbf{E}_k$ into query, key, & value vectors, respectively.

> **3. Attention Score Calculation (Purple Box)**
* For each token $k$, the attention scores $\mathbf{S}_{k,j}$ between the current token $k$ & all previous tokens $j$ (including itself) are computed by taking the **dot product** of the **query vector** $\mathbf{Q}_k$ of token $k$ with the **key vector** $\mathbf{K}_j$ of token $j$:
    * $\mathbf{S}_{k,j} = \frac{\mathbf{Q}_k \cdot \mathbf{K}_j^T}{\sqrt{D}}$
* $D$ is the dimension of the key vectors, & the division by $\sqrt{D}$ is done to ensure that the dot product values do not grow too large.
* These attention scores $\mathbf{S}_{k,j}$ represent how much token $k$ should attend to each previous token $j$.

> **4. Masking in Self-Attention**
* Since this is **masked self-attention**, the attention scores for **future tokens** are masked out, ensuring that each token only attends to previous tokens & itself.
* For example, token $\mathbf{T_k}$ can only attend to tokens $\mathbf{T_1}, \mathbf{T_2}, \dots, \mathbf{T_k}$, but not tokens $\mathbf{T_{k+1}}, \mathbf{T_{k+2}}, \dots$.

> **5. Attention Weights Calculation (Green Box)**
* After calculating the attention scores, the scores are passed through a **softmax function** to compute the **attention weights** $\mathbf{A}_{k,j}$:
    * $\mathbf{A}_\text{k,j} = \text{softmax}(\mathbf{S}_\text{k,j})$
* The softmax function ensures that the attention scores are normalized & sum to 1, so that the final attention weights represent the relative importance of each token $j$ for generating token $k$'s context vector.

> **6. Context Vector Calculation (Red Box)**
* The **context vector** $\mathbf{C}_k$ for each token $k$ is computed as a **weighted sum** of the **value vectors** $\mathbf{V}_j$, where the weights are the attention weights $\mathbf{A}_{k,j}$:
    * $\mathbf{C}_k = \sum_\text{j=1}^{k} (\mathbf{A}_\text{k,j} \cdot \mathbf{V}_j)$
* The context vector $\mathbf{C}_k$ is a combination of all the other tokens’ value vectors (up to token $k$), weighted by how much attention token $k$ gives to each previous token $j$.
* This means that the context vector for each token $k$ captures information from all previous tokens in the sequence, weighted by their relevance as determined by the attention mechanism.

> **7. Autoregressive Nature of the Decoder**
* In the **masked self-attention head**, each token can only attend to **previous tokens** & itself, which is why it is referred to as **autoregressive**.
* The context vectors $\mathbf{C}_1, \mathbf{C}_2, \dots, \mathbf{C}_k, \dots, \mathbf{C}_{255}$ are generated **sequentially** during the decoding process, meaning that:
    * $\mathbf{C}_1$ is used to generate $\mathbf{T_2}$,
    * $\mathbf{C}_2$ is used to generate $\mathbf{T_3}$,
    * $\mathbf{C}_k$ is used to generate $\mathbf{T}_\text{k+1}$, & so on.
* This process continues until the model generates all 256 tokens or reaches an **end-of-sequence (EOS)** token.


### **1.4. Cross Attention Head in Decoder**

<div align="left">
  <img src="https://github.com/mhrafiei/figures/blob/main/535_743/module_12/cross_attention_head.png?raw=true" width="150%">
  <br>
  <figcaption>Figure: Cross-Attention Head Example</figcaption>
</div>

The figure represents the **cross-attention head** in an autoregressive manner, showing how the decoder context vectors interact with the encoder context vectors to compute attention during text generation.

> **1. Masked Self-Attention Head Context Vectors (Left Side)**
* On the left, you see the **masked self-attention head context vectors** from the decoder for tokens:
    * $\mathbf{U}_1, \mathbf{U}_2, \dots, \mathbf{U}_k, \dots, \mathbf{U}_\text{255}$.
* These vectors are the outputs of the masked self-attention heads from the previous decoder layers.
* Each context vector $\mathbf{U}_k$ corresponds to a token at the $k$-th position in the partially generated output sequence (up to token $T_k$).

> **2. Masked Self-Attention Head Query Vectors (Green Box)**
* The masked self-attention context vectors are transformed into **query vectors** $\mathbf{Q}_k$ for each token.
    * $\mathbf{Q}_k = \mathbf{W}_\text{query} \cdot \mathbf{U}_k$
* This is done using a learned weight matrix $\mathbf{W}_\text{query}$.
* The query vectors $\mathbf{Q}_k$ are used to attend to the **encoder context vectors** to capture the relationship between the partially generated sequence (decoder output) & the input sequence (encoder output).

> **3. Encoder Context Vectors (Blue Box)**
* The **encoder context vectors** $\mathbf{E}_1, \mathbf{E}_2, \dots, \mathbf{E}_{11}$ represent the embeddings from the encoder corresponding to each input token in the source sequence.
    * These vectors are the result of the encoder's processing of the input sequence.
* The encoder context vectors are used to produce **keys** $\mathbf{K}_i$ & **values** $\mathbf{V}_i$ via linear transformations:
    * $\mathbf{K}_i = \mathbf{W}_\text{key} \cdot \mathbf{E}_i$
    * $\mathbf{V}_i = \mathbf{W}_\text{value} \cdot \mathbf{E}_i$
* $\mathbf{W}_{\text{key}}$ & $\mathbf{W}_{\text{value}}$ are learned weight matrices.

> **4. Attention Score Calculation (Purple Box)**
* The attention scores $\mathbf{S}_{k,i}$ between the query vector $\mathbf{Q}_k$ from the decoder & the key vector $\mathbf{K}_i$ from the encoder are calculated using the dot product:
    * $\mathbf{S}_{k,i} = \frac{\mathbf{Q}_k \cdot \mathbf{K}_i^T}{\sqrt{D}}$
* $D$ is the dimensionality of the key vectors, & the division by $\sqrt{D}$ is done to normalize the dot product values.
* The attention scores $\mathbf{S}_{k,i}$ determine how much attention the current token $T_k$ in the decoder gives to each token in the input sequence (encoder output).

> **5. Attention Weights Calculation (Green Box)**
* The attention scores are passed through a **softmax function** to compute the **attention weights** $\mathbf{A}_\text{k,i}$:
    * $\mathbf{A}_\text{k,i} = \text{softmax}(\mathbf{S}_\text{k,i})$
* The softmax function ensures that the attention scores are normalized & sum to 1. The attention weights determine how much focus token $T_k$ should give to each token $T_i$ from the input sequence.

> **6. Context Vector Calculation (Red Box)**
* The **context vector** $\mathbf{C}_k$ for each token $T_k$ in the decoder is computed as a **weighted sum** of the **value vectors** $\mathbf{V}_i$ from the encoder, where the weights are the attention weights $\mathbf{A}_\text{k,i}$:
    * $\mathbf{C}_k = \sum_\text{i=1}^{11} (\mathbf{A}_\text{k,i} \cdot \mathbf{V}_i)$
* The context vector $\mathbf{C}_k$ combines information from all input tokens, weighted by the attention given to each token $T_i$ in the input sequence.
* This context vector captures the relationship between the current decoder token $T_k$ & the input sequence, & it is passed on to further operations (e.g., feed-forward network or next layers).


### **1.5. Decoder Architecture**

<div align="left">
  <img src="https://github.com/mhrafiei/figures/blob/main/535_743/module_12/decoder.png?raw=true" width="150%">
  <br>
  <figcaption>Figure: Decoder Architecture</figcaption>
</div>

The figure represents the decoder process in a Transformer model, focusing on how tokens are processed through both **masked self-attention** & **cross-attention** mechanisms, with residual connections, normalization, & feed-forward layers. The encoder provides context vectors from the input sequence, & the decoder generates a sequence of output tokens.

> **1. Decoder Input: Encoder Context Vectors (Left Side of the Figure)**
* The **encoder context vectors** are provided to the decoder. These vectors represent the encoded input sequence, with one context vector for each input token.
* Each context vector from the encoder is of size $1 \times 512$, for example:
    * $\mathbf{E}_1, \mathbf{E}_2, \dots, \mathbf{E}_\text{11}$ for 11 tokens in the input sequence.

> **2. Masked Self-Attention (First Block)**
* The **masked self-attention** mechanism ensures that each token in the decoder can only attend to itself & the tokens generated before it, preventing access to "future" tokens during the generation process.

> **2.1 Splitting for Masked Self-Attention Heads (Blue Box)**
* Each token's embedding in the decoder (size $1 \times 512$) is **split** into $N_\text{MSA} = 8$ smaller vectors, one for each masked self-attention head. MSA stands for Masked Self-Attention.
* For example, for token $T_k$, its embedding $\mathbf{E}_{T_k}$ is split into $\mathbf{E}_{\mathbf{T}_k, 1}, \mathbf{E}_{\mathbf{T}_k, 2}, \dots, \mathbf{E}_{\mathbf{T}_k, 8}$ each of size $1 \times 64$.

>> **2.2 Masked Self-Attention Heads**
* Each of the 8 **masked self-attention heads** computes **attention scores** & **context vectors** for the token $T_k$, but only attends to tokens generated before $T_k$.
* The output of each masked self-attention head is a **context vector** of size $1 \times 64$.

>> **2.3 Concatenation (Green Box)**
* The 8 context vectors from the masked self-attention heads are **concatenated** to form a single vector of size $1 \times 512$, combining information from all heads: $\mathbf{C}_{\mathbf{T}_k} = \text{Concatenate}(\mathbf{C}_{\mathbf{T}_k,1}, \mathbf{C}_{\mathbf{T}_k,2}, \dots, \mathbf{C}_{\mathbf{T}_k,8})$

>> **2.4 Residual Connection & Normalization (Purple Box)**
* A **residual connection** is applied by adding the original input embedding to the concatenated output from the self-attention heads: $\mathbf{E}_\text{{residual, msa}} = \mathbf{E}_{\mathbf{T}_k} + \mathbf{C}_\text{{concat, msa}}$
* **Layer normalization** is applied to stabilize the output, ensuring more efficient training.

> **3. Cross-Attention (Second Block)**
* In the **cross-attention** mechanism, each token in the decoder attends to the encoder context vectors, allowing the decoder to use information from the input sequence when generating each token.

>> **3.1 Splitting for Cross-Attention Heads (Blue Box)**
* The output of the masked self-attention heads is **split** again into $N_\text{CA} = 8$ smaller vectors, one for each cross-attention head.
* For token $\mathbf{T}_k$, the normalized output from the masked self-attention is split into: $\mathbf{E}_{\mathbf{T}_k, 1}^{\text{norm}}, \mathbf{E}_{\mathbf{T_k}, 2}^{\text{norm}}, \dots, \mathbf{E}_{\mathbf{T}_k, 8}^{\text{norm}}$.

>> **3.2 Cross-Attention Heads**
* Each of the 8 **cross-attention heads** computes **attention scores** between the token’s embedding & the encoder’s context vectors.
* The heads generate **context vectors** by attending to the input tokens, each outputting a vector of size $1 \times 64$.

>> **3.3 Concatenation (Green Box)**
* The 8 cross-attention context vectors are **concatenated** to form a vector of size $1 \times 512$ for each token.

>> **3.4 Residual Connection & Normalization (Purple Box)**
* A **residual connection** is applied by adding the normalized output of the masked self-attention to the concatenated output from the cross-attention heads: $\mathbf{E}_\text{{residual, CA}} = \mathbf{E}_\text{{norm, msa}} + \mathbf{C}_\text{{concat, ca}}$
* **Layer normalization** is applied again to stabilize the output.

> **4. Fully Connected Layer (Orange Box)**
* After the cross-attention, the output is passed through a **fully connected feed-forward network (FFN)**, which consists of two linear transformations with a non-linearity (e.g., ReLU) in between:
    * First, a linear transformation maps the normalized vector to a higher-dimensional space:
        * $\mathbf{Z}_1 = \mathbf{W}_1 \cdot \mathbf{E}_\text{{normalized, CA}} + \mathbf{b}_1$
    * Then, a non-linear activation function is applied:
        * $\mathbf{Z}_2 = \text{ReLU}(\mathbf{Z}_1)$
    * Finally, another linear transformation maps the output back to the original dimensionality:
        * $\mathbf{E}_\text{{FFN}} = \mathbf{W}_2 \cdot \mathbf{Z}_2 + \mathbf{b}_2$

> **5. Residual Connection & Normalization (Second Residual Path)**
* After the feed-forward network, another **residual connection** is applied by adding the input to the fully connected layer to its output:
    * $\mathbf{E}_\text{{residual, FFN}} = \mathbf{E}_\text{{normalized, CA}} + \mathbf{E}_\text{{FFN}}$
* **Layer normalization** is applied to the result.


## **2. Advantages of Models with Attention Mechanism**

### **2.1. Enhancing Context Awareness & Coherence**

Traditional models like LSTMs process sequences in a fixed order, maintaining a hidden state that carries information from previous time steps. While LSTMs are capable of handling short-term dependencies, they struggle with long-term dependencies due to issues like vanishing gradients. The Attention mechanism addresses this limitation by creating direct connections between all words in the input & output sequences, regardless of their positions.

This enhancement in context awareness leads to more accurate & coherent text generation. For mechanical engineering applications, such as writing detailed reports or generating design specifications, the ability to maintain context over long passages is crucial. Attention allows the model to reference relevant parts of the input text dynamically, ensuring that generated content is both accurate & contextually appropriate.

### **2.2. Improving Text Generation Tasks in Mechanical Engineering**

In mechanical engineering, precise language & technical accuracy are paramount. Whether drafting a report on system performance or generating specifications for a new component, engineers require tools that can handle complex terminology & intricate relationships between concepts.

Compared to traditional methods like LSTMs, models utilizing Attention can:

* **Handle Longer Texts:** By attending to all parts of the input, models can manage longer documents without losing context.
* **Capture Complex Dependencies:** Attention enables models to understand relationships between distant words, which is essential for technical documents where critical information may be spread throughout the text.
* **Improve Relevance & Accuracy:** By focusing on relevant input segments, the model’s output is more likely to be accurate & aligned with the intended message.

### **2.3. Deep Dive into Multi-Head Attention**

Building upon the basic Attention mechanism, Multi-Head Attention is a sophisticated technique used in Transformer architectures. It allows the model to attend to different parts of the input simultaneously, capturing various levels of dependencies & patterns.

Multi-Head Attention involves multiple attention mechanisms (or “heads”) running in parallel. Each head operates in its own subspace, allowing the model to:

* **Capture Diverse Features:** Different heads can focus on different types of relationships, such as syntactic structure or semantic meaning.
* **Model Multiple Dependencies:** In complex sentences, words may have multiple relationships. Multi-Head Attention can capture these simultaneously.
* **Enhance Learning Capacity:** By aggregating information from multiple attention heads, the model can learn richer representations of the input data.

For mechanical engineering tasks that involve complex text generation, such as:

* **Generating Technical Documentation:** Accurately describing intricate systems & processes.
* **Describing Machine Behavior in Real-Time:** Interpreting sensor data & generating descriptive reports.

Multi-Head Attention provides the flexibility & accuracy needed to handle these challenges. By considering multiple aspects of the input data concurrently, models can produce outputs that are both detailed & contextually appropriate.

## **3. Large Language Models & the Transformer Architecture**

Large Language Models (LLMs), such as GPT (Generative Pre-trained Transformers), have leveraged the Transformer architecture to achieve state-of-the-art performance in various NLP tasks.

Transformers rely heavily on the Attention mechanism, particularly Self-Attention, where the model attends to different positions of the same sequence to compute a representation of the sequence.

> Key components of the Transformer architecture include:
* **Encoder & Decoder Layers:** Stacks of layers that process the input & generate the output.
* **Positional Encoding:** Since Transformers do not process sequences in order, positional encodings are added to represent the position of words in the sequence.
* **Feed-Forward Networks:** Applied after the attention layers to transform the attended representations.

### **3.1. Advantages Over Traditional Models**

Transformers & LLMs offer several advantages over models like LSTMs:

* **Parallelization:** Transformers process all words in a sequence simultaneously, allowing for faster computation & training on large datasets.
* **Handling Long Dependencies:** With Attention mechanisms, Transformers can capture relationships between distant words effectively.
* **Scalability:** LLMs can be scaled up by increasing the number of layers & parameters, leading to improved performance.

### **3.2. Applications in Mechanical Engineering**

For mechanical engineers, LLMs can be powerful tools in:

* **Generating Research Summaries:** Condensing technical papers into digestible summaries.
* **Creating Mechanical Part Descriptions:** Automatically generating detailed descriptions based on specifications.
* **Automating Technical Reports:** Compiling data & observations into comprehensive reports.

By leveraging the Transformer architecture, these models can understand & generate long, complex texts typical in mechanical engineering literature, surpassing the capabilities of older methods like LSTMs & SVMs.

## **4. LLM Tasks, Architectures, Training Data, & Major Models**

Large Language Models (LLMs) based on Transformer architectures have revolutionized the field of Natural Language Processing (NLP) by excelling in a wide array of tasks. Understanding the different tasks, the appropriate Transformer architectures for each, & the structure of their training data is crucial for effectively leveraging these models.

### **4.1. Transformer Architectures Overview**

The following table outlines the three main architectures used in transformer models—Encoder-Only, Decoder-Only, & Encoder-Decoder—along with their primary tasks & notable models. Encoder-Only models, such as BERT & RoBERTa, are best suited for tasks like text classification, named entity recognition, sentiment analysis, & question answering (extractive). Decoder-Only models, such as GPT-2, GPT-3, & GPT-4, excel at language modeling, text generation, dialogue systems, & code generation, where they generate coherent text based on prompts. Encoder-Decoder models, including T5 & BART, are designed for tasks such as machine translation, text summarization, paraphrase generation, & generative question answering. Each architecture is optimized for specific types of natural language processing tasks, demonstrating the flexibility of transformer models across diverse applications.

<table border="1" cellspacing="0" cellpadding="5">
  <thead>
    <tr>
      <th><strong>Transformer Architecture</strong></th>
      <th><strong>Primary Tasks</strong></th>
      <th><strong>Example Models</strong></th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><strong>Encoder-Only</strong></td>
      <td>
        <ul>
          <li>Text Classification</li>
          <li>Named Entity Recognition (NER)</li>
          <li>Sentiment Analysis</li>
          <li>Question Answering (Extractive)</li>
          <li>Textual Entailment</li>
          <li>Coreference Resolution</li>
        </ul>
      </td>
      <td>
        <a href="https://arxiv.org/abs/1810.04805" target="_blank">BERT</a>,
        <a href="https://arxiv.org/abs/1907.11692" target="_blank">RoBERTa</a>,
        <a href="https://arxiv.org/abs/1909.11942" target="_blank">ALBERT</a>,
        <a href="https://arxiv.org/abs/1910.01108" target="_blank">DistilBERT</a>,
        <a href="https://arxiv.org/abs/1907.10529" target="_blank">SpanBERT</a>
      </td>
    </tr>
    <tr>
      <td><strong>Decoder-Only</strong></td>
      <td>
        <ul>
          <li>Language Modeling</li>
          <li>Text Generation</li>
          <li>Dialogue Systems</li>
          <li>Text Completion</li>
          <li>Code Generation</li>
        </ul>
      </td>
      <td>
        <a href="https://arxiv.org/abs/1901.04506" target="_blank">GPT-2</a>,
        <a href="https://arxiv.org/abs/2005.14165" target="_blank">GPT-3</a>,
        <a href="https://openai.com/research/gpt-4" target="_blank">GPT-4</a>,
        <a href="https://arxiv.org/abs/2205.01068" target="_blank">OPT</a>,
        <a href="https://arxiv.org/abs/2211.05100" target="_blank">BLOOM</a>,
        <a href="https://arxiv.org/abs/1911.00536" target="_blank">DialoGPT</a>
      </td>
    </tr>
    <tr>
      <td><strong>Encoder-Decoder</strong></td>
      <td>
        <ul>
          <li>Machine Translation</li>
          <li>Text Summarization</li>
          <li>Paraphrase Generation</li>
          <li>Question Answering (Generative)</li>
          <li>Text Infilling</li>
          <li>Dialogue Systems (Structured)</li>
        </ul>
      </td>
      <td>
        <a href="https://arxiv.org/abs/1910.10683" target="_blank">T5</a>,
        <a href="https://arxiv.org/abs/1910.13461" target="_blank">BART</a>,
        <a href="https://arxiv.org/abs/2001.08210" target="_blank">MarianMT</a>,
        <a href="https://arxiv.org/abs/2004.13637" target="_blank">mBART</a>,
        <a href="https://arxiv.org/abs/1912.08777" target="_blank">Pegasus</a>
      </td>
    </tr>
  </tbody>
</table>

### **4.2. List of Transformer-Based NLP Tasks**

The following table provides an overview of various Transformer-based NLP tasks, describing their goals, relevant architectures, training data structures, & notable models. For **Language Modeling**, Decoder-Only architectures like GPT-2 & GPT-3 are used to predict the next token in a sequence, with training data structured as `(input_sequence, next_token)`. **Text Generation** also uses a Decoder-Only model, generating text based on a prompt, as seen in GPT series & BLOOM. **Machine Translation**, using an Encoder-Decoder architecture, translates text from one language to another, with models like T5 & MarianMT. **Text Summarization**, another Encoder-Decoder task, involves condensing documents, with models like T5 & Pegasus leading the way. For tasks like **Text Classification** & **Named Entity Recognition (NER)**, Encoder-Only architectures such as BERT & RoBERTa are used. Models like BERT & T5 are also applied to **Question Answering** & **Sentiment Analysis** tasks, where understanding context or determining sentiment is critical. **Paraphrase Generation** & **Dialogue Systems** (chatbots) are covered by Encoder-Decoder or Decoder-Only models, generating responses or reformulating sentences. **Code Generation** uses models like Codex to produce programming code from descriptions. Tasks such as **Speech Recognition** & **Text-to-Speech (TTS)** utilize Encoder-Decoder models, while **Multi-Task Learning** leverages versatile models like T5 for handling multiple NLP tasks simultaneously. Finally, **Zero-Shot/Few-Shot** models, such as GPT-3, tackle tasks without explicit training for them, showcasing the flexibility of Transformer architectures across a range of NLP challenges.

<table border="1" cellspacing="0" cellpadding="5">
  <thead>
    <tr>
      <th><strong>Task</strong></th>
      <th><strong>Description</strong></th>
      <th><strong>Transformer Architecture</strong></th>
      <th><strong>Training Data Structure</strong></th>
      <th><strong>Notable Models</strong></th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><strong>Language Modeling</strong></td>
      <td>Predicting the next word/token in a sequence.</td>
      <td>Decoder-Only</td>
      <td>(input_sequence, next_token)</td>
      <td>
        <a href="https://arxiv.org/abs/1901.04506" target="_blank">GPT-2</a>,
        <a href="https://arxiv.org/abs/2005.14165" target="_blank">GPT-3</a>,
        <a href="https://openai.com/research/gpt-4" target="_blank">GPT-4</a>
      </td>
    </tr>
    <tr>
      <td><strong>Text Generation</strong></td>
      <td>Generating coherent & contextually relevant text based on a prompt.</td>
      <td>Decoder-Only</td>
      <td>(prompt, generated_text)</td>
      <td>
        <a href="https://arxiv.org/abs/1901.04506" target="_blank">GPT Series</a>,
        <a href="https://arxiv.org/abs/2205.01068" target="_blank">OPT</a>,
        <a href="https://arxiv.org/abs/2211.05100" target="_blank">BLOOM</a>
      </td>
    </tr>
    <tr>
      <td><strong>Machine Translation</strong></td>
      <td>Translating text from one language to another.</td>
      <td>Encoder-Decoder</td>
      <td>((source_sentence, decoder_input), target_translation)</td>
      <td>
        <a href="https://arxiv.org/abs/1910.10683" target="_blank">T5</a>,
        <a href="https://arxiv.org/abs/2001.08210" target="_blank">MarianMT</a>,
        <a href="https://arxiv.org/abs/2001.08210" target="_blank">mBART</a>,
        <a href="https://arxiv.org/abs/1910.13461" target="_blank">BART</a>
      </td>
    </tr>
    <tr>
      <td><strong>Text Summarization</strong></td>
      <td>Creating a concise summary of a longer text document.</td>
      <td>Encoder-Decoder</td>
      <td>(document, summary)</td>
      <td>
        <a href="https://arxiv.org/abs/1910.10683" target="_blank">T5</a>,
        <a href="https://arxiv.org/abs/1910.13461" target="_blank">BART</a>,
        <a href="https://arxiv.org/abs/1912.08777" target="_blank">Pegasus</a>
      </td>
    </tr>
    <tr>
      <td><strong>Text Classification</strong></td>
      <td>Assigning predefined categories or labels to text data.</td>
      <td>Encoder-Only</td>
      <td>(input_text, label)</td>
      <td>
        <a href="https://arxiv.org/abs/1810.04805" target="_blank">BERT</a>,
        <a href="https://arxiv.org/abs/1907.11692" target="_blank">RoBERTa</a>,
        <a href="https://arxiv.org/abs/1909.11942" target="_blank">ALBERT</a>,
        <a href="https://arxiv.org/abs/1910.01108" target="_blank">DistilBERT</a>
      </td>
    </tr>
    <tr>
      <td><strong>Named Entity Recognition (NER)</strong></td>
      <td>Identifying & classifying named entities in text (e.g., persons, organizations, locations).</td>
      <td>Encoder-Only</td>
      <td>(input_text, entity_labels)</td>
      <td>
        <a href="https://arxiv.org/abs/1810.04805" target="_blank">BERT</a>,
        <a href="https://arxiv.org/abs/1907.11692" target="_blank">RoBERTa</a>,
        <a href="https://spacy.io/" target="_blank">SpaCy</a> (not Transformer-based but relevant)
      </td>
    </tr>
    <tr>
      <td><strong>Question Answering (QA)</strong></td>
      <td>Providing answers to questions based on a given context or document.</td>
      <td>Encoder-Decoder or Encoder-Only</td>
      <td>(question, context, answer_span) or ((question + context), answer)</td>
      <td>
        <a href="https://arxiv.org/abs/1810.04805" target="_blank">BERT</a> (for extractive QA),
        <a href="https://arxiv.org/abs/1910.10683" target="_blank">T5</a>,
        <a href="https://arxiv.org/abs/1910.13461" target="_blank">BART</a>,
        <a href="https://arxiv.org/abs/2003.10555" target="_blank">ELECTRA</a>
      </td>
    </tr>
    <tr>
      <td><strong>Sentiment Analysis</strong></td>
      <td>Determining the sentiment expressed in a piece of text (e.g., positive, negative, neutral).</td>
      <td>Encoder-Only</td>
      <td>(input_text, sentiment_label)</td>
      <td>
        <a href="https://arxiv.org/abs/1810.04805" target="_blank">BERT</a>,
        <a href="https://arxiv.org/abs/1907.11692" target="_blank">RoBERTa</a>,
        <a href="https://arxiv.org/abs/1906.08237" target="_blank">XLNet</a>,
        <a href="https://arxiv.org/abs/1910.01108" target="_blank">DistilBERT</a>
      </td>
    </tr>
    <tr>
      <td><strong>Paraphrase Generation</strong></td>
      <td>Generating a paraphrased version of a given sentence or text.</td>
      <td>Encoder-Decoder</td>
      <td>(original_sentence, paraphrased_sentence)</td>
      <td>
        <a href="https://arxiv.org/abs/1910.10683" target="_blank">T5</a>,
        <a href="https://arxiv.org/abs/1910.13461" target="_blank">BART</a>,
        <a href="https://arxiv.org/abs/1912.08777" target="_blank">Pegasus</a>
      </td>
    </tr>
    <tr>
      <td><strong>Dialogue Systems (Chatbots)</strong></td>
      <td>Engaging in conversational exchanges with users, generating appropriate responses.</td>
      <td>Decoder-Only or Encoder-Decoder</td>
      <td>(dialogue_history, response)</td>
      <td>
        <a href="https://arxiv.org/abs/2005.14165" target="_blank">GPT-3</a>,
        <a href="https://arxiv.org/abs/2004.13637" target="_blank">BlenderBot</a>,
        <a href="https://arxiv.org/abs/1911.00536" target="_blank">DialoGPT</a>,
        <a href="https://arxiv.org/abs/1910.10683" target="_blank">T5</a>
      </td>
    </tr>
    <tr>
      <td><strong>Text Completion</strong></td>
      <td>Completing a partially written sentence or paragraph.</td>
      <td>Decoder-Only</td>
      <td>(partial_text, completed_text)</td>
      <td>
        <a href="https://arxiv.org/abs/1901.04506" target="_blank">GPT Series</a>,
        <a href="https://arxiv.org/abs/1910.10683" target="_blank">T5</a>
      </td>
    </tr>
    <tr>
      <td><strong>Text-to-Speech (TTS)</strong></td>
      <td>Converting written text into spoken words.</td>
      <td>Encoder-Decoder (with audio)</td>
      <td>(input_text, audio_waveform)</td>
      <td>
        <a href="https://arxiv.org/abs/1712.05884" target="_blank">Tacotron</a>,
        <a href="https://arxiv.org/abs/1706.03762" target="_blank">Transformer TTS</a> (various implementations)
      </td>
    </tr>
    <tr>
      <td><strong>Speech Recognition</strong></td>
      <td>Transcribing spoken language into written text.</td>
      <td>Encoder-Decoder or Encoder-Only</td>
      <td>(audio_waveform, transcribed_text)</td>
      <td>
        <a href="https://arxiv.org/abs/2109.01652" target="_blank">Whisper (by OpenAI)</a>,
        <a href="https://arxiv.org/abs/2006.11477" target="_blank">Wav2Vec 2.0</a>,
        <a href="https://arxiv.org/abs/1706.03762" target="_blank">Speech Transformers</a>
      </td>
    </tr>
    <tr>
      <td><strong>Code Generation</strong></td>
      <td>Generating code snippets or entire programs based on a description or prompt.</td>
      <td>Decoder-Only or Encoder-Decoder</td>
      <td>(code_prompt, code_snippet)</td>
      <td>
        <a href="https://arxiv.org/abs/2107.03374" target="_blank">Codex</a> (GPT-3 based),
        <a href="https://arxiv.org/abs/2201.10098" target="_blank">CodeT5</a>,
        <a href="https://www.deepmind.com/blog/alphacode-writing-competitive-programming-code" target="_blank">AlphaCode</a>
      </td>
    </tr>
    <tr>
      <td><strong>Text Infilling</strong></td>
      <td>Filling in missing parts of a text with contextually appropriate words or phrases.</td>
      <td>Encoder-Decoder</td>
      <td>(partial_text, complete_text)</td>
      <td>
        <a href="https://arxiv.org/abs/1910.13461" target="_blank">BART</a>,
        <a href="https://arxiv.org/abs/1910.10683" target="_blank">T5</a>,
        <a href="https://arxiv.org/abs/1912.08777" target="_blank">PEGASUS</a>
      </td>
    </tr>
    <tr>
      <td><strong>Coreference Resolution</strong></td>
      <td>Identifying when different expressions refer to the same entity in text.</td>
      <td>Encoder-Only</td>
      <td>(input_text, coreference_labels)</td>
      <td>
        <a href="https://arxiv.org/abs/1907.10529" target="_blank">SpanBERT</a>,
        <a href="https://arxiv.org/abs/1810.04805" target="_blank">BERT</a>,
        <a href="https://arxiv.org/abs/1907.11692" target="_blank">RoBERTa</a>
      </td>
    </tr>
    <tr>
      <td><strong>Semantic Role Labeling</strong></td>
      <td>Assigning roles to words or phrases in a sentence to represent their semantic relationships.</td>
      <td>Encoder-Only</td>
      <td>(input_text, semantic_roles)</td>
      <td>
        <a href="https://arxiv.org/abs/1810.04805" target="_blank">BERT</a>,
        <a href="https://arxiv.org/abs/1907.11692" target="_blank">RoBERTa</a>,
        <a href="https://arxiv.org/abs/1906.08237" target="_blank">XLNet</a>
      </td>
    </tr>
    <tr>
      <td><strong>Textual Entailment</strong></td>
      <td>Determining if one sentence logically follows from another.</td>
      <td>Encoder-Only</td>
      <td>(premise, hypothesis, entailment_label)</td>
      <td>
        <a href="https://arxiv.org/abs/1810.04805" target="_blank">BERT</a>,
        <a href="https://arxiv.org/abs/1907.11692" target="_blank">RoBERTa</a>,
        <a href="https://arxiv.org/abs/1909.11942" target="_blank">ALBERT</a>,
        <a href="https://arxiv.org/abs/1906.08237" target="_blank">XLNet</a>
      </td>
    </tr>
    <tr>
      <td><strong>Language Inference</strong></td>
      <td>Inferring missing information or logical conclusions from given text.</td>
      <td>Encoder-Only</td>
      <td>(input_text, inference)</td>
      <td>
        <a href="https://arxiv.org/abs/1810.04805" target="_blank">BERT</a>,
        <a href="https://arxiv.org/abs/1907.11692" target="_blank">RoBERTa</a>,
        <a href="https://arxiv.org/abs/1910.10683" target="_blank">T5</a>,
        <a href="https://arxiv.org/abs/1910.13461" target="_blank">BART</a>
      </td>
    </tr>
    <tr>
      <td><strong>Multi-Task Learning</strong></td>
      <td>Training models to perform multiple NLP tasks simultaneously.</td>
      <td>Varies</td>
      <td>Depends on tasks; generally multi-input multi-output</td>
      <td>
        <a href="https://arxiv.org/abs/1910.10683" target="_blank">T5</a>,
        <a href="https://arxiv.org/abs/2207.05391" target="_blank">MT5</a>,
        <a href="https://arxiv.org/abs/2010.11934" target="_blank">UnifiedQA</a>,
        <a href="https://arxiv.org/abs/2209.02107" target="_blank">T0</a>
      </td>
    </tr>
    <tr>
      <td><strong>Zero-Shot / Few-Shot Tasks</strong></td>
      <td>Performing tasks with little to no task-specific training data by leveraging pre-trained knowledge.</td>
      <td>Varies</td>
      <td>Depends on task; often formatted as prompt-response pairs</td>
      <td>
        <a href="https://arxiv.org/abs/2005.14165" target="_blank">GPT-3</a>,
        <a href="https://arxiv.org/abs/1910.10683" target="_blank">T5</a>,
        <a href="https://arxiv.org/abs/2301.00396" target="_blank">FLAN</a>
      </td>
    </tr>
  </tbody>
</table>

### **4.3. Detailed Descriptions & Data Structures**


The following table provides a comprehensive breakdown of various Transformer-based Natural Language Processing (NLP) tasks, with a focus on their descriptions, suitable architectures, training data structures, & practical examples relevant to mechanical engineering. The tasks range from **Language Modeling** using a Decoder-Only architecture, where the goal is to predict the next token in a sequence, to more complex tasks such as **Machine Translation**, which utilizes an Encoder-Decoder architecture to translate technical language between different languages. It also includes tasks such as **Text Summarization**, **Named Entity Recognition**, **Question Answering**, & **Code Generation**, each demonstrating how these tasks can be adapted to mechanical engineering contexts, like summarizing reports on additive manufacturing or generating Python functions for engineering calculations. The table shows how these architectures (Decoder-Only, Encoder-Only, & Encoder-Decoder) apply to different tasks & training data structures, highlighting the versatility of Transformers in automating & enhancing various technical tasks across engineering disciplines.


<table border="1" cellspacing="0" cellpadding="5">
  <thead>
    <tr>
      <th><strong>Task</strong></th>
      <th><strong>Description</strong></th>
      <th><strong>Transformer Architecture</strong></th>
      <th><strong>Training Data Structure</strong></th>
      <th><strong>Example Relevant to Mechanical Engineering</strong></th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><strong>Language Modeling</strong></td>
      <td>Predicting the next word/token in a sequence.</td>
      <td>Decoder-Only</td>
      <td>(input_sequence, next_token)</td>
      <td>
        *Input:* "The stress distribution in the beam increases with the"  
        *Target:* "load applied."
      </td>
    </tr>
    <tr>
      <td><strong>Machine Translation</strong></td>
      <td>Translating text from one language to another.</td>
      <td>Encoder-Decoder</td>
      <td>((source_sentence, decoder_input), target_translation)</td>
      <td>
        *Input:* "The torque generated by the engine can be calculated using"  
        *Target:* "La torsión generada por el motor se puede calcular utilizando"
      </td>
    </tr>
    <tr>
      <td><strong>Text Summarization</strong></td>
      <td>Creating a concise summary of a longer text document.</td>
      <td>Encoder-Decoder</td>
      <td>(document, summary)</td>
      <td>
        *Input:* "Additive manufacturing, commonly known as 3D printing, allows for the creation of complex geometries that are difficult to achieve with traditional manufacturing methods. This technology is revolutionizing the production processes in various industries, including aerospace & automotive."  
        *Target:* "3D printing enables complex designs & is transforming manufacturing in aerospace & automotive sectors."
      </td>
    </tr>
    <tr>
      <td><strong>Text Classification</strong></td>
      <td>Assigning predefined categories or labels to text data.</td>
      <td>Encoder-Only</td>
      <td>(input_text, label)</td>
      <td>
        *Input:* "The finite element analysis revealed high stress concentrations at the weld joints."  
        *Label:* "Structural Analysis"
      </td>
    </tr>
    <tr>
      <td><strong>Named Entity Recognition (NER)</strong></td>
      <td>Identifying & classifying named entities in text (e.g., persons, organizations, locations).</td>
      <td>Encoder-Only</td>
      <td>(input_text, entity_labels)</td>
      <td>
        *Input:* "The newly developed turbine by GE Aviation has improved efficiency by 15%."  
        *Target:* `[("GE Aviation", "ORG"), ("turbine", "PRODUCT"), ("15%", "PERCENT")]`
      </td>
    </tr>
    <tr>
      <td><strong>Question Answering (QA)</strong></td>
      <td>Providing answers to questions based on a given context or document.</td>
      <td>Encoder-Decoder or Encoder-Only</td>
      <td>(question, context, answer_span) or ((question + context), answer)</td>
      <td>
        *Input:*  
        - **Question:** "What is the maximum tensile strength of the material used in the suspension system?"  
        - **Context:** "The suspension system utilizes AISI 4140 steel, which has a maximum tensile strength of 655 MPa."  
        *Target:* "655 MPa"
      </td>
    </tr>
    <tr>
      <td><strong>Sentiment Analysis</strong></td>
      <td>Determining the sentiment expressed in a piece of text (e.g., positive, negative, neutral).</td>
      <td>Encoder-Only</td>
      <td>(input_text, sentiment_label)</td>
      <td>
        *Input:* "The new CAD software significantly enhances the design workflow, making it much more efficient."  
        *Label:* "Positive"
      </td>
    </tr>
    <tr>
      <td><strong>Paraphrase Generation</strong></td>
      <td>Generating a paraphrased version of a given sentence or text.</td>
      <td>Encoder-Decoder</td>
      <td>(original_sentence, paraphrased_sentence)</td>
      <td>
        *Input:* "The compressor's efficiency decreases as the temperature rises."  
        *Target:* "As the temperature increases, the compressor becomes less efficient."
      </td>
    </tr>
    <tr>
      <td><strong>Dialogue Systems (Chatbots)</strong></td>
      <td>Engaging in conversational exchanges with users, generating appropriate responses.</td>
      <td>Decoder-Only or Encoder-Decoder</td>
      <td>(dialogue_history, response)</td>
      <td>
        *Input:*  
        - **User:** "Can you explain how the heat exchanger works in the HVAC system?"  
        - **Bot:**  
        *Target:* "Certainly! A heat exchanger in an HVAC system transfers heat between two or more fluids without mixing them, helping to regulate temperature efficiently."
      </td>
    </tr>
    <tr>
      <td><strong>Code Generation</strong></td>
      <td>Generating code snippets or entire programs based on a description or prompt.</td>
      <td>Decoder-Only or Encoder-Decoder</td>
      <td>(code_prompt, code_snippet)</td>
      <td>
        *Input:* "Write a Python function to calculate the Reynolds number for fluid flow."  
        *Target:*
        ```python
        def calculate_reynolds_number(rho, v, L, mu):
            """
            Calculate the Reynolds number.
            Parameters:
            rho (float): Density of the fluid (kg/m³)
            v (float): Velocity of the fluid (m/s)
            L (float): Characteristic length (m)
            mu (float): Dynamic viscosity (Pa·s)
            Returns:
            float: Reynolds number
            """
            return (rho * v * L) / mu
        ```
      </td>
    </tr>
    <tr>
      <td><strong>Text Infilling</strong></td>
      <td>Filling in missing parts of a text with contextually appropriate words or phrases.</td>
      <td>Encoder-Decoder</td>
      <td>(partial_text, complete_text)</td>
      <td>
        *Input:* "The [MASK] system ensures that the vehicle maintains stability during turns."  
        *Target:* "The suspension system ensures that the vehicle maintains stability during turns."
      </td>
    </tr>
    <tr>
      <td><strong>Coreference Resolution</strong></td>
      <td>Identifying when different expressions refer to the same entity in text.</td>
      <td>Encoder-Only</td>
      <td>(input_text, coreference_labels)</td>
      <td>
        *Input:* "The gearbox was tested extensively. It showed excellent performance under stress."  
        *Target:* `[("The gearbox", "It")]`
      </td>
    </tr>
    <tr>
      <td><strong>Semantic Role Labeling</strong></td>
      <td>Assigning roles to words or phrases in a sentence to represent their semantic relationships.</td>
      <td>Encoder-Only</td>
      <td>(input_text, semantic_roles)</td>
      <td>
        *Input:* "The engineer designed the hydraulic system to improve efficiency."  
        *Target:* `[("engineer", "Agent"), ("hydraulic system", "Theme"), ("improve", "Goal")]`
      </td>
    </tr>
    <tr>
      <td><strong>Textual Entailment</strong></td>
      <td>Determining if one sentence logically follows from another.</td>
      <td>Encoder-Only</td>
      <td>(premise, hypothesis, entailment_label)</td>
      <td>
        *Input:*  
        - **Premise:** "The robotic arm is capable of high-precision movements."  
        - **Hypothesis:** "The robotic arm can move accurately."  
        *Label:* "Entailment"
      </td>
    </tr>
    <tr>
      <td><strong>Language Inference</strong></td>
      <td>Inferring missing information or logical conclusions from given text.</td>
      <td>Encoder-Only</td>
      <td>(input_text, inference)</td>
      <td>
        *Input:* "The material properties were optimized for high-temperature applications."  
        *Target:* "The materials used can withstand elevated temperatures without degrading."
      </td>
    </tr>
    <tr>
      <td><strong>Multi-Task Learning</strong></td>
      <td>Training models to perform multiple NLP tasks simultaneously.</td>
      <td>Varies</td>
      <td>Depends on tasks; generally multi-input multi-output</td>
      <td>
        *Example:* A model trained to both classify the type of mechanical failure & generate a maintenance report based on sensor data.
      </td>
    </tr>
    <tr>
      <td><strong>Zero-Shot / Few-Shot Tasks</strong></td>
      <td>Performing tasks with little to no task-specific training data by leveraging pre-trained knowledge.</td>
      <td>Varies</td>
      <td>Depends on task; often formatted as prompt-response pairs</td>
      <td>
        *Example:* Prompting the model to predict the optimal gear ratio for a new vehicle design without explicit training on gear ratio optimization tasks.
      </td>
    </tr>
  </tbody>
</table>

### **4.4. Examples of Popular Transformer Models & Their Applications**

The following table provides a comprehensive comparison of several prominent Transformer-based models, detailing their architectures, primary applications, & key features. For example, **BERT**, an Encoder-Only model, excels at tasks like text classification, named entity recognition (NER), & sentiment analysis due to its bidirectional context understanding. **GPT-3**, a Decoder-Only model, is known for its capabilities in text generation, dialogue systems, & code generation, leveraging few-shot & zero-shot learning. **T5** & **BART**, both Encoder-Decoder models, are versatile in tasks such as machine translation, text summarization, & paraphrase generation. Specialized models like **Codex** focus on code generation & programming assistance, while **Whisper** targets speech recognition & transcription. Models like **AlphaCode** are tailored for competitive programming & algorithm development, & **FLAN** stands out with its ability to handle zero-shot & few-shot learning tasks, demonstrating enhanced generalization across diverse applications. Each model offers distinct strengths based on its architecture & pre-training approach, making them well-suited for a variety of NLP & engineering tasks.


<table border="1" cellspacing="0" cellpadding="5">
  <thead>
    <tr>
      <th><strong>Model</strong></th>
      <th><strong>Architecture</strong></th>
      <th><strong>Primary Applications</strong></th>
      <th><strong>Key Features</strong></th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><strong>BERT</strong></td>
      <td>Encoder-Only</td>
      <td>
        <ul>
          <li>Text Classification</li>
          <li>Named Entity Recognition (NER)</li>
          <li>Sentiment Analysis</li>
          <li>Question Answering (Extractive)</li>
        </ul>
      </td>
      <td>
        <ul>
          <li>Bidirectional context understanding</li>
          <li>Pre-trained on masked language modeling</li>
          <li>Fine-tuned for specific tasks</li>
        </ul>
      </td>
    </tr>
    <tr>
      <td><strong>GPT-3</strong></td>
      <td>Decoder-Only</td>
      <td>
        <ul>
          <li>Text Generation</li>
          <li>Dialogue Systems</li>
          <li>Code Generation</li>
          <li>Language Modeling</li>
        </ul>
      </td>
      <td>
        <ul>
          <li>Large-scale autoregressive model</li>
          <li>Few-shot & zero-shot learning capabilities</li>
          <li>Extensive knowledge base from pre-training</li>
        </ul>
      </td>
    </tr>
    <tr>
      <td><strong>T5</strong></td>
      <td>Encoder-Decoder</td>
      <td>
        <ul>
          <li>Machine Translation</li>
          <li>Text Summarization</li>
          <li>Paraphrase Generation</li>
          <li>Question Answering (Generative)</li>
        </ul>
      </td>
      <td>
        <ul>
          <li>Text-to-text framework for versatility</li>
          <li>Multi-task learning capabilities</li>
          <li>Unified approach to diverse NLP tasks</li>
        </ul>
      </td>
    </tr>
    <tr>
      <td><strong>BART</strong></td>
      <td>Encoder-Decoder</td>
      <td>
        <ul>
          <li>Text Summarization</li>
          <li>Paraphrase Generation</li>
          <li>Text Infilling</li>
          <li>Machine Translation</li>
        </ul>
      </td>
      <td>
        <ul>
          <li>Combines bidirectional & autoregressive training</li>
          <li>Effective for generation & comprehension tasks</li>
          <li>Pre-trained with denoising autoencoder objectives</li>
        </ul>
      </td>
    </tr>
    <tr>
      <td><strong>MarianMT</strong></td>
      <td>Encoder-Decoder</td>
      <td>
        <ul>
          <li>Machine Translation</li>
          <li>Multilingual Applications</li>
        </ul>
      </td>
      <td>
        <ul>
          <li>Supports numerous language pairs</li>
          <li>Designed for efficient translation tasks</li>
          <li>Open-source & community-driven</li>
        </ul>
      </td>
    </tr>
    <tr>
      <td><strong>DialoGPT</strong></td>
      <td>Decoder-Only</td>
      <td>
        <ul>
          <li>Conversational AI</li>
          <li>Chatbots</li>
          <li>Dialogue Systems</li>
        </ul>
      </td>
      <td>
        <ul>
          <li>Fine-tuned for dialogue generation</li>
          <li>Enhanced response coherence & relevance</li>
          <li>Pre-trained on large-scale conversational data</li>
        </ul>
      </td>
    </tr>
    <tr>
      <td><strong>Codex</strong></td>
      <td>Decoder-Only</td>
      <td>
        <ul>
          <li>Code Generation</li>
          <li>Programming Assistance</li>
          <li>Automated Coding Tasks</li>
        </ul>
      </td>
      <td>
        <ul>
          <li>Specialized for programming languages</li>
          <li>Integrated with IDEs for real-time assistance</li>
          <li>Extensive knowledge of coding syntax & libraries</li>
        </ul>
      </td>
    </tr>
    <tr>
      <td><strong>Whisper</strong></td>
      <td>Encoder-Decoder</td>
      <td>
        <ul>
          <li>Speech Recognition</li>
          <li>Transcription Services</li>
          <li>Voice Command Interpretation</li>
        </ul>
      </td>
      <td>
        <ul>
          <li>Multilingual speech-to-text capabilities</li>
          <li>Robust against various accents & noise levels</li>
          <li>Pre-trained on diverse audio datasets</li>
        </ul>
      </td>
    </tr>
    <tr>
      <td><strong>CodeT5</strong></td>
      <td>Encoder-Decoder</td>
      <td>
        <ul>
          <li>Code Generation</li>
          <li>Code Summarization</li>
          <li>Code Translation</li>
        </ul>
      </td>
      <td>
        <ul>
          <li>Specialized for programming languages</li>
          <li>Fine-tuned on code-related tasks</li>
          <li>Supports multiple programming languages</li>
        </ul>
      </td>
    </tr>
    <tr>
      <td><strong>AlphaCode</strong></td>
      <td>Decoder-Only</td>
      <td>
        <ul>
          <li>Competitive Programming</li>
          <li>Algorithm Development</li>
          <li>Automated Problem Solving</li>
        </ul>
      </td>
      <td>
        <ul>
          <li>Designed for high-performance code generation</li>
          <li>Trained on competitive programming datasets</li>
          <li>Optimized for solving complex engineering problems</li>
        </ul>
      </td>
    </tr>
    <tr>
      <td><strong>FLAN</strong></td>
      <td>Varies</td>
      <td>
        <ul>
          <li>Zero-Shot & Few-Shot Learning Tasks</li>
          <li>Multi-Task Applications</li>
        </ul>
      </td>
      <td>
        <ul>
          <li>Fine-tuned for instruction following</li>
          <li>Enhanced generalization across tasks</li>
          <li>Supports a wide range of engineering-related queries</li>
        </ul>
      </td>
    </tr>
  </tbody>
</table>


### **4.5. Best Practices for Preparing Training Data**

The following table outlines best practices for preparing training data for transformer models, with specific considerations for mechanical engineering tasks. It emphasizes the importance of consistent formatting across datasets, ensuring balanced representation of various categories, & using high-quality annotations for accurate model training. Practices like data cleaning & managing domain-specific terminology are crucial, especially for handling the technical jargon common in mechanical engineering. Other key points include avoiding data leakage, utilizing metadata, & augmenting data to improve model robustness. The table also highlights the need for annotation consistency, privacy protection for sensitive data, & scalable data pipelines to manage large volumes of engineering-related information. Lastly, ongoing quality assurance processes are recommended to maintain the relevance & accuracy of the training data, particularly in rapidly evolving fields like mechanical engineering.

<table border="1" cellspacing="0" cellpadding="5">
  <thead>
    <tr>
      <th><strong>Best Practice</strong></th>
      <th><strong>Description</strong></th>
      <th><strong>Mechanical Engineering Consideration</strong></th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><strong>Consistent Formatting</strong></td>
      <td>Ensure that input-output pairs are consistently structured across the dataset.</td>
      <td>
        *Standardize the format of technical documents, such as maintaining consistent terminology in design specifications & reports.*
      </td>
    </tr>
    <tr>
      <td><strong>Balanced Datasets</strong></td>
      <td>Ensure all classes or categories are adequately represented to prevent model bias.</td>
      <td>
        *For classification tasks, include a balanced number of examples from different engineering categories like "Thermal Analysis," "Structural Design," & "Fluid Dynamics."*
      </td>
    </tr>
    <tr>
      <td><strong>High-Quality Annotations</strong></td>
      <td>Accurate labeling of data is crucial for model performance, especially in tasks requiring precision.</td>
      <td>
        *In NER tasks, precisely label components, materials, & standards in technical texts to improve entity recognition accuracy.*
      </td>
    </tr>
    <tr>
      <td><strong>Diverse & Representative Samples</strong></td>
      <td>Include varied examples to help the model generalize well to unseen data.</td>
      <td>
        *Incorporate documents from different mechanical engineering domains, such as automotive, aerospace, & manufacturing, to enhance model versatility.*
      </td>
    </tr>
    <tr>
      <td><strong>Data Cleaning</strong></td>
      <td>Remove noise, correct typos, & standardize text to enhance training quality.</td>
      <td>
        *Clean engineering reports by removing irrelevant sections, correcting nomenclature errors, & ensuring consistent units of measurement.*
      </td>
    </tr>
    <tr>
      <td><strong>Tokenization Alignment</strong></td>
      <td>Use the same tokenizer for both input & target sequences to maintain consistency in token representation.</td>
      <td>
        *Ensure that technical terms, symbols, & abbreviations used in mechanical engineering are properly tokenized to prevent misinterpretation by the model.*
      </td>
    </tr>
    <tr>
      <td><strong>Handling Technical Jargon</strong></td>
      <td>Properly manage domain-specific terminology to ensure the model understands & processes specialized language.</td>
      <td>
        *Include a comprehensive glossary of mechanical engineering terms & ensure they are consistently used across the dataset.*
      </td>
    </tr>
    <tr>
      <td><strong>Avoiding Data Leakage</strong></td>
      <td>Ensure that information from the target is not inadvertently included in the input.</td>
      <td>
        *In translation tasks, make sure the target language data does not appear in the source language input to maintain task integrity.*
      </td>
    </tr>
    <tr>
      <td><strong>Augmenting Data</strong></td>
      <td>Use data augmentation techniques to expand the dataset & introduce variability.</td>
      <td>
        *Generate paraphrased versions of technical instructions or vary the phrasing of engineering problems to increase data diversity.*
      </td>
    </tr>
    <tr>
      <td><strong>Metadata Utilization</strong></td>
      <td>Incorporate additional metadata to enrich the context & improve model understanding.</td>
      <td>
        *Include metadata such as document type (e.g., report, specification), engineering domain, & versioning to provide contextual cues for the model.*
      </td>
    </tr>
    <tr>
      <td><strong>Annotation Consistency</strong></td>
      <td>Maintain consistency in annotations to prevent confusion & improve learning efficiency.</td>
      <td>
        *Use standardized labeling schemes for components & processes in mechanical engineering to ensure uniformity across the dataset.*
      </td>
    </tr>
    <tr>
      <td><strong>Privacy & Confidentiality</strong></td>
      <td>Ensure that sensitive or proprietary engineering data is handled appropriately to maintain confidentiality.</td>
      <td>
        *Anonymize sensitive project details & proprietary designs in technical documents before using them for training.*
      </td>
    </tr>
    <tr>
      <td><strong>Scalability</strong></td>
      <td>Prepare data pipelines that can handle large volumes of engineering data efficiently.</td>
      <td>
        *Implement automated scripts to preprocess & format extensive engineering databases, CAD files, & simulation results for seamless integration into training pipelines.*
      </td>
    </tr>
    <tr>
      <td><strong>Quality Assurance</strong></td>
      <td>Regularly review & validate the training data to ensure ongoing quality & relevance.</td>
      <td>
        *Conduct periodic audits of the dataset to remove outdated information, correct inconsistencies, & incorporate the latest engineering standards & practices.*
      </td>
    </tr>
  </tbody>
</table>


## **5. Fine-Tuning Large Language Models for Specific Domains**

While pre-trained LLMs are powerful, fine-tuning them for specific domains like mechanical engineering can significantly enhance their performance on specialized tasks.

### **4.1. Introduction to LoRA (Low-Rank Adaptation)**

[LoRA](https://arxiv.org/pdf/2106.09685) [and also, its later version, [QLoRA](https://arxiv.org/pdf/2305.14314)] is a fine-tuning technique that adapts pre-trained LLMs to new tasks by introducing a small number of trainable parameters, reducing computational & storage requirements.

> How LoRA Works:
* **Parameter Efficiency:** Instead of updating all the parameters of the LLM, LoRA adds low-rank matrices to the model’s weights, which are trained during fine-tuning.
* **Resource Optimization:** This approach minimizes the computational resources needed, making it feasible to fine-tune large models on limited hardware.
* **Maintaining Pre-Trained Knowledge:** By only adjusting a subset of parameters, the model retains its general language understanding while adapting to the new domain.

Using LoRA, engineers can:
* **Specialize Models in Specific Disciplines:** Fine-tune models for areas like thermodynamics or fluid mechanics without extensive computational resources.
* **Improve Domain-Specific Accuracy:** Tailor the model’s vocabulary & understanding to include technical terms & concepts unique to mechanical engineering.
* **Accelerate Development:** Quickly adapt models for new tasks or data without retraining from scratch.

More:
* https://www.youtube.com/watch?v=t1caDsMzWBk
* https://www.youtube.com/watch?v=t509sv5MT0w&t=438s


### **4.2. Other Fine-Tuning Techniques**

Beyond LoRA, other fine-tuning methods applicable to mechanical engineering tasks include:

* **Prompt Tuning:** Adjusting the input prompts to guide the model’s output towards desired content.
* **Adapter Modules:** Adding small modules to the network that can be trained separately, allowing for domain adaptation without altering the main model.
* **Knowledge Distillation:** Transferring knowledge from a large model to a smaller one, retaining performance while reducing size.

These techniques enable the development of specialized models for tasks like:

* **Predicting System Failures:** Analyzing historical data to forecast potential issues.
* **Generating Summaries for Complex Systems:** Creating concise overviews of large mechanical systems.

## **6. Build Transformers in Python**

### **6.1. Libraries & Environment Setup**

In [ ]:
#@title Install SentencePiece Library

# The 'pip install' command is used to install Python libraries.
# 'sentencepiece' is a library designed for unsupervised text tokenization & is often used in natural language processing (NLP).
# It is particularly useful for training & using subword-based tokenizers, which help models handle out-of-vocabulary words & improve
# language model performance by splitting rare words into common subword units.

# This installation command will download & install SentencePiece from PyPI (Python Package Index).
!pip install sentencepiece

In [ ]:
#@title Import Libraries
# Importing PyTorch libraries for building & training neural networks:
# - 'torch' is the core PyTorch library, providing data structures & functions for tensor computations.
# - 'torch.nn' contains modules & classes for creating various neural network layers & architectures.
# - 'torch.optim' includes optimization algorithms such as SGD & Adam for training neural network models.
import torch
import torch.nn as nn
import torch.optim as optim

# Importing the SentencePiece library for text tokenization.
# SentencePiece is a tokenizer & text processor widely used in NLP for subword-based tokenization.
import sentencepiece as spm

# Importing standard Python libraries:
# - 'numpy' is used for numerical operations, such as array manipulation & mathematical calculations.
# - 'os' is used for interacting with the operating system, e.g., file handling & directory operations.
# - 'random' is used to generate random numbers & shuffle data for training.
# - 'time' is used to measure execution time & introduce delays in processes.
# - 'sys' is used for interacting with the Python runtime environment, such as adding paths or handling I/O.
# - 'glob' is used for pattern matching & retrieving file paths, useful for batch processing files.
import numpy as np
import os
import random
import time
import sys
import glob

# The string module in Python provides various functions & constants that are helpful for handling text data.
import string

# Importing type hints for specifying argument & return types in functions.
from typing import *

# Importing PyTorch learning rate scheduler:
# 'LambdaLR' is used to adjust the learning rate according to a custom function during training.
from torch.optim.lr_scheduler import LambdaLR

# Importing PyTorch utilities for handling data:
# 'Dataset' is an abstract class for representing datasets. It is used when creating custom data loaders.
# 'DataLoader' provides an iterable over a dataset, enabling easy batching & shuffling of data.
from torch.utils.data import Dataset, DataLoader

# Importing 'tqdm' for creating progress bars in Jupyter Notebooks.
# This makes it easier to monitor the progress of training & evaluation loops in an interactive way.
from tqdm.notebook import tqdm

# Importing a utility from IPython to clear notebook output.
# This helps keep the output clean by removing previous outputs in the Jupyter Notebook.
from IPython.display import clear_output

# Mounting Google Drive for file access in Google Colab.
# This allows for easy loading & saving of models & data stored in Google Drive.
from google.colab import drive
drive.mount('/content/drive')

### **6.2. Model Functions & Classes**

In [ ]:
#@title Positional Encoding
class PositionalEncoding(nn.Module):
    """
    Positional encoding module for adding position information to input embeddings.
    This helps the model understand the relative positions of tokens in a sequence.
    Transformers, unlike RNNs, do not inherently understand sequence order, so this module
    provides essential positional information.
    More info on transformers: https://arxiv.org/abs/1706.03762
    """
    def __init__(self, d_model: int, max_len: int = 5000):
        # Call the parent class constructor to initialize the nn.Module.
        super(PositionalEncoding, self).__init__()

        # Create a zero tensor to store the positional encodings with dimensions (max_len, d_model).
        # This tensor will later be filled with calculated positional values.
        pe = torch.zeros(max_len, d_model).to(device)

        # Generate a tensor that represents positions from 0 to max_len-1.
        # `unsqueeze(1)` reshapes this tensor from (max_len) to (max_len, 1) to enable broadcasting during calculations.
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1).to(device)

        # Calculate the scaling factor for the sinusoidal position encoding formula:
        # 10000^(2i/d_model). The `arange` method generates even indices (0, 2, 4, ...) to be used for sine & cosine functions.
        # For a detailed explanation on positional encoding, refer to section 3.5 in https://arxiv.org/abs/1706.03762
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model)).to(device)

        # Assign sine values to even indices (0, 2, 4, ...) in the positional encoding matrix.
        # `position * div_term` scales the positions by the calculated factor to match the sinusoidal pattern.
        pe[:, 0::2] = torch.sin(position * div_term)

        # Assign cosine values to odd indices (1, 3, 5, ...) in the positional encoding matrix.
        pe[:, 1::2] = torch.cos(position * div_term)

        # Add an extra dimension to the positional encoding tensor.
        # This makes the shape (1, max_len, d_model), which is necessary for batch processing during training.
        self.pe = pe.unsqueeze(0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass to add positional encoding to input embeddings.

        Args:
        - x (Tensor): Tensor of input embeddings with shape (batch_size, seq_len, d_model), where batch_size is the
                      number of sequences processed simultaneously, seq_len is the length of the sequence, and
                      d_model is the embedding dimension.

        Returns:
        - Tensor: Tensor with positional encoding added, maintaining the shape (batch_size, seq_len, d_model).
        """
        # Add the precomputed positional encodings to the input embeddings, up to the length of the input sequence.
        # This operation helps the model differentiate tokens based on their position within the sequence.
        x = x + self.pe[:, :x.size(1), :]
        return x


In [ ]:
#@title Transformer Language Model

class TransformerLanguageModel(nn.Module):
    """
    Transformer-based language model for sequence generation.
    This model is designed to handle language modeling tasks by predicting the next tokens in a sequence.
    It follows the Transformer architecture, which is known for its effectiveness in handling sequences
    & parallel computation.
    Refer to the original Transformer paper for more details: https://arxiv.org/abs/1706.03762
    """
    def __init__(self, vocab_size: int, d_model: int, num_heads: int, num_layers: int, dff: int, dropout_rate: float):
        # Initialize the parent nn.Module class.
        super(TransformerLanguageModel, self).__init__()

        # Define the type of model for clarity.
        self.model_type = 'Transformer'
        self.d_model = d_model  # The dimensionality of the model's embeddings & hidden states.

        # Token embedding layer:
        # Maps input tokens (integers) to continuous vector representations of size `d_model`.
        # Embeddings allow the model to represent words in a high-dimensional vector space.
        self.embedding = nn.Embedding(vocab_size, d_model)

        # Positional encoding:
        # Adds positional information to the token embeddings so that the model can understand
        # the order of tokens in the sequence. This is essential because the Transformer architecture
        # itself does not inherently consider token positions.
        # For more on positional encodings & their importance, see: https://jalammar.github.io/illustrated-transformer/
        self.pos_encoder = PositionalEncoding(d_model, max_len=max_seq_length)

        # Transformer encoder stack:
        # Consists of `num_layers` encoder layers, each with multi-head self-attention & feed-forward layers.
        # `d_model` specifies the dimension of each attention head's input, `num_heads` is the number of attention heads,
        # `dff` is the size of the feed-forward network hidden layer, & `dropout_rate` is for regularization.
        # The Transformer encoder is the core that processes the input data using self-attention mechanisms.
        encoder_layer = nn.TransformerEncoderLayer(d_model, num_heads, dff, dropout_rate)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers)

        # Final linear layer:
        # Maps the output of the encoder (with `d_model` dimensions) to the vocabulary size.
        # This step is necessary for converting model outputs into token probabilities for prediction.
        self.fc_out = nn.Linear(d_model, vocab_size)

        # Dropout layer for regularization to prevent overfitting during training.
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, src: torch.Tensor, src_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Perform a forward pass through the Transformer model.

        Args:
        - src (Tensor): The input sequence of token IDs, shape (batch_size, seq_len).
        - src_mask (Tensor, optional): A mask tensor that prevents attention to certain positions,
          typically used to mask future tokens during training.

        Returns:
        - logits (Tensor): Output predictions with shape (batch_size, seq_len, vocab_size),
          where each entry represents the probability distribution over the vocabulary for each token.
        """
        # Step 1: Token embedding.
        # Convert input tokens to embeddings & scale them by the square root of `d_model`.
        # This scaling helps with the stability of gradients early in training.
        # Reference: The scaling factor is discussed in the original Transformer paper (Vaswani et al., 2017).
        src = self.embedding(src) * np.sqrt(self.d_model)

        # Step 2: Add positional encoding to the embeddings to encode the sequence order.
        # Positional encoding helps the model understand the position of each word in the sequence.
        src = self.pos_encoder(src)

        # Step 3: Apply dropout to help prevent overfitting.
        src = self.dropout(src)

        # Step 4: Transpose the input to match the expected shape for the Transformer encoder:
        # (seq_len, batch_size, d_model). This is necessary as the Transformer expects input with the
        # time dimension (sequence length) as the first dimension.
        src = src.transpose(0, 1)

        # Step 5: Pass the input through the Transformer encoder.
        # The encoder applies multi-head attention & feed-forward layers sequentially.
        # Multi-head attention allows the model to attend to different parts of the sequence simultaneously.
        output = self.transformer_encoder(src, mask=src_mask)

        # Step 6: Transpose the output back to the original format:
        # (batch_size, seq_len, d_model). This step ensures compatibility with the final linear layer.
        output = output.transpose(0, 1)

        # Step 7: Apply the final linear layer to project the outputs to the vocabulary size.
        # This prepares the model for computing predictions. Each position in the output sequence
        # is mapped to a vector of size `vocab_size`, representing probabilities for each token.
        logits = self.fc_out(output)

        return logits


In [ ]:
#@title Generate Square Subsequent Mask
def generate_square_subsequent_mask(sz: int) -> torch.Tensor:
    """
    Generate a square mask for a sequence to be used in transformer models.
    This mask ensures that each position in the sequence can only attend to
    current & previous positions, preventing future information from being accessed.

    This is important in tasks like language modeling, where predictions are made
    one token at a time, & each token should only consider preceding tokens to
    avoid data leakage.
    For more about attention masks & their role in Transformers, refer to:
    https://jalammar.github.io/illustrated-transformer/

    Args:
    - sz (int): The size of the mask, representing both the number of rows & columns.
      This corresponds to the sequence length.

    Returns:
    - mask (torch.Tensor): A boolean mask tensor of shape (sz, sz). Positions
      above the main diagonal are True (indicating masked/future positions),
      & those on & below the diagonal are False (indicating allowed positions).
    """
    # Create an upper triangular matrix filled with ones above the main diagonal,
    # which will act as a mask for future positions. Positions on the diagonal
    # & below are filled with zeros (unmasked).
    # `torch.triu` creates the upper triangular matrix, & `diagonal=1` ensures
    # that masking starts right above the main diagonal.
    mask = torch.triu(torch.ones(sz, sz), diagonal=1).bool()

    # Transfer the mask to the same device (CPU or GPU) as the rest of the model
    # to ensure compatibility during training or inference.
    # This step is crucial when training on GPU to prevent device mismatches.
    mask = mask.to(device)

    # Return the mask, which is used in transformer layers to prevent attention
    # from looking ahead at future positions in the input sequence.
    # This masking mechanism is essential for autoregressive tasks where only past
    # information should be utilized.
    return mask


In [ ]:
#@title Custom Learning Rate Scheduler
def lr_lambda(step: int) -> float:
    """
    Custom learning rate scheduler function for a Transformer model.
    Implements a warm-up phase followed by a decay phase as described in the
    original Transformer paper (Vaswani et al., 2017). This helps in stabilizing
    training & improving convergence.

    The learning rate follows the formula:
    lr = (d_model ^ -0.5) * min(step_num ^ -0.5, step_num * (warmup_steps ^ -1.5))

    Args:
    - step (int): The current training step (iteration).

    Returns:
    - float: The learning rate scaling factor for the given step.

    For more on the original paper & learning rate schedules, refer to:
    https://arxiv.org/abs/1706.03762
    """
    # The scaling factor for the learning rate is based on the model's embedding dimension `d_model`.
    # This scaling helps with normalizing the gradient flow as the model size grows.
    # (d_model ^ -0.5) is a constant multiplier applied throughout training.
    # The constant factor ensures that the learning rate is scaled appropriately for the model's size.
    constant_factor = d_model ** -0.5

    # Calculate the two terms that determine the learning rate:
    # 1. Inverse square root decay: (step_num ^ -0.5)
    #    - Ensures that after the warm-up period, the learning rate decreases proportionally to 1/sqrt(step).
    #    - This phase supports a stable decrease in learning rate as training progresses.
    decay_phase = (step + 1) ** -0.5

    # 2. Warm-up phase: (step_num * (warmup_steps ^ -1.5))
    #    - During the initial warm-up period, the learning rate increases linearly.
    #    - This prevents the model from converging too quickly & helps to stabilize early training.
    #    - The warm-up is crucial for better gradient flow & to avoid exploding gradients early on.
    warmup_phase = (step + 1) * (warmup_steps ** -1.5)

    # Return the product of the constant scaling factor & the minimum of the two terms
    # to implement the warm-up followed by decay.
    # By taking the minimum, we allow the learning rate to increase during the warm-up phase
    # & then switch to the decay phase after warm-up is complete.
    return constant_factor * min(decay_phase, warmup_phase)

In [ ]:
#@title One Epoch Training

def train_epoch(model: nn.Module, optimizer: optim.Optimizer, scheduler: LambdaLR, dataloader: DataLoader) -> Tuple[float, float]:
    """
    Train the model for one epoch & compute average loss & accuracy.

    Args:
    - model (nn.Module): The Transformer model to be trained.
    - optimizer (optim.Optimizer): The optimizer used for updating model parameters.
    - scheduler (LambdaLR): The learning rate scheduler for dynamic adjustment of learning rate.
    - dataloader (DataLoader): DataLoader that provides batches of training data.

    Returns:
    - Tuple[float, float]: A tuple containing the average loss & accuracy for the epoch.
    """
    # Set the model to training mode. This enables dropout & other training-specific layers.
    model.train()
    # Initialize variables to accumulate the total loss & count correct predictions for accuracy.
    total_loss = 0
    total_correct = 0
    total_tokens = 0

    # Initialize tqdm progress bar for training to visualize the progress of the epoch.
    # `leave=False` ensures the progress bar is removed after completion.
    progress_bar = tqdm(dataloader, desc="Training", leave=False)

    # Iterate over each batch in the dataloader.
    for batch_idx, (input_seq, target_seq) in enumerate(progress_bar):
        # Transfer input & target sequences to the designated device (CPU or GPU).
        input_seq = input_seq.to(device)
        target_seq = target_seq.to(device)

        # Reset the gradients from the previous step.
        optimizer.zero_grad()
        # Generate a mask for the input sequence to prevent attention to future tokens.
        # This is important for autoregressive tasks like language modeling.
        src_mask = generate_square_subsequent_mask(input_seq.size(1))
        # Forward pass through the model using the input sequence & the generated mask.
        output = model(input_seq, src_mask=src_mask)
        # Reshape the output to have shape (batch_size * seq_len, vocab_size) for loss calculation.
        output = output.view(-1, vocab_size)
        # Flatten the target sequence for the same reason.
        target = target_seq.view(-1)
        # Compute the loss between the model's predictions & the target.
        # The criterion should be defined with a suitable loss function, e.g., CrossEntropyLoss.
        loss = criterion(output, target)
        # Perform backpropagation to compute gradients.
        loss.backward()
        # Apply gradient clipping to prevent exploding gradients during training.
        # `max_norm=1.0` sets the threshold for clipping.
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        # Update model parameters based on computed gradients.
        optimizer.step()
        # Step the learning rate scheduler to adjust the learning rate.
        scheduler.step()
        # Accumulate the current batch loss to calculate the average loss later.
        total_loss += loss.item()

        # Compute the number of correct predictions for accuracy calculation.
        # `torch.argmax(output, dim=1)` gets the predicted class indices.
        predicted_tokens = torch.argmax(output, dim=1)
        # Sum up the number of correctly predicted tokens.
        correct_predictions = (predicted_tokens == target).sum().item()
        total_correct += correct_predictions
        # Count the total number of tokens processed.
        total_tokens += target.size(0)

        # Calculate the loss & accuracy for the current batch for monitoring.
        batch_loss = loss.item()
        batch_accuracy = correct_predictions / target.size(0)

        # Update the tqdm progress bar with current batch loss & accuracy.
        progress_bar.set_postfix({'Batch Loss': f'{batch_loss:.4f}', 'Batch Accuracy': f'{batch_accuracy:.4f}'})

    # Calculate average loss over all batches for the epoch.
    avg_loss = total_loss / len(dataloader)
    # Calculate accuracy over all tokens processed during the epoch.
    accuracy = total_correct / total_tokens
    # Return the average loss & accuracy for reporting purposes.
    return avg_loss, accuracy

In [ ]:
#@title Evaluation
def evaluate(model: nn.Module, dataloader: DataLoader) -> Tuple[float, float]:
    """
    Evaluate the model on the validation or test dataset.

    Args:
    - model (nn.Module): The Transformer model to be evaluated.
    - dataloader (DataLoader): DataLoader that provides batches of evaluation data.

    Returns:
    - Tuple[float, float]: A tuple containing the average loss & accuracy for the evaluation.
    """
    # Set the model to evaluation mode. This disables dropout & other training-specific behaviors.
    model.eval()
    # Initialize variables to accumulate total loss & correct predictions for accuracy.
    total_loss = 0
    total_correct = 0
    total_tokens = 0

    # Initialize tqdm progress bar for evaluation to visualize progress.
    # `leave=False` ensures the progress bar is removed after completion.
    progress_bar = tqdm(dataloader, desc="Evaluating", leave=False)

    # Use `torch.no_grad()` to disable gradient computation during evaluation.
    # This reduces memory usage & speeds up computation.
    with torch.no_grad():
        # Iterate over each batch in the dataloader.
        for batch_idx, (input_seq, target_seq) in enumerate(progress_bar):
            # Transfer input & target sequences to the designated device (CPU or GPU).
            input_seq = input_seq.to(device)
            target_seq = target_seq.to(device)
            # Generate a mask for the input sequence to ensure that attention does not
            # look at future tokens during the sequence processing.
            src_mask = generate_square_subsequent_mask(input_seq.size(1))
            # Forward pass through the model to get the output predictions.
            output = model(input_seq, src_mask=src_mask)
            # Reshape the output to have shape (batch_size * seq_len, vocab_size) for loss calculation.
            output = output.view(-1, vocab_size)
            # Flatten the target sequence for compatibility with the output.
            target = target_seq.view(-1)
            # Compute the loss between the model's predictions & the target.
            # The criterion should be defined as a suitable loss function, e.g., CrossEntropyLoss.
            loss = criterion(output, target)
            # Accumulate the total loss for calculating average loss later.
            total_loss += loss.item()

            # Compute the number of correct predictions for accuracy calculation.
            # `torch.argmax(output, dim=1)` returns the predicted token indices for each position.
            predicted_tokens = torch.argmax(output, dim=1)
            # Count how many predictions match the actual target values.
            correct_predictions = (predicted_tokens == target).sum().item()
            # Accumulate the total number of correct predictions.
            total_correct += correct_predictions
            # Count the total number of tokens processed for accuracy computation.
            total_tokens += target.size(0)

            # Calculate the loss & accuracy for the current batch for monitoring.
            batch_loss = loss.item()
            batch_accuracy = correct_predictions / target.size(0)

            # Update the tqdm progress bar with the current batch loss & accuracy.
            progress_bar.set_postfix({'Batch Loss': f'{batch_loss:.4f}', 'Batch Accuracy': f'{batch_accuracy:.4f}'})

    # Calculate the average loss over all batches for the epoch.
    avg_loss = total_loss / len(dataloader)
    # Calculate the overall accuracy across all processed tokens.
    accuracy = total_correct / total_tokens
    # Return the average loss & accuracy for reporting purposes.
    return avg_loss, accuracy


### **6.3. Data Management Functions**

In [ ]:
#@title Data Functions
def text_to_ids(text: str) -> list:
    """
    Convert a given text input into a list of token IDs using a pre-trained SentencePiece tokenizer.
    This function is essential for preparing text data for input to NLP models, where input must be
    represented as numerical data (token IDs).

    Args:
    - text (str): The input text string to be tokenized.

    Returns:
    - list: A list of token IDs representing the input text.

    For more information on tokenization & SentencePiece, refer to:
    https://github.com/google/sentencepiece & the original paper: https://arxiv.org/abs/1804.10959
    """
    # Encode the input text into token IDs using the SentencePiece tokenizer's `EncodeAsIds` method.
    # This method transforms text into a list of integers that represent each token.
    return sp.EncodeAsIds(text)

def ids_to_text(ids: list) -> str:
    """
    Convert a list of token IDs back into a human-readable text using a pre-trained SentencePiece tokenizer.
    This function reverses the tokenization process, allowing model outputs (predicted IDs) to be interpreted
    as natural language.

    Args:
    - ids (list): A list of token IDs to be converted back into text.

    Returns:
    - str: A decoded string representing the original text.

    This step is crucial for interpreting model outputs in tasks like language generation.
    """
    # Decode the list of token IDs back into a text string using the SentencePiece tokenizer's `DecodeIds` method.
    # This helps translate model-generated IDs into human-readable text, facilitating evaluation & display.
    return sp.DecodeIds(ids)

In [ ]:
#@title TextDataset Classes
class TextDataset(Dataset):
    """
    Custom Dataset for text data.
    This dataset is designed to prepare text sequences for training language models. It takes in raw text data,
    tokenizes it, & creates input-target sequence pairs suitable for training transformer-based models.

    Args:
    - text_data (List[str]): A list of text strings to be tokenized & converted into training sequences.
    - max_seq_length (int): The maximum length of each input sequence. This determines how much context the model
                            receives for each training example.

    For more on how text data is processed for training language models, refer to:
    https://huggingface.co/transformers/preprocessing.html
    """
    def __init__(self, text_data: List[str], max_seq_length: int):
        # Initialize an empty list to store the input-target sequence pairs.
        self.sequences = []

        # Iterate over each text in the provided text data.
        for text in text_data:
            # Tokenize the current text using a pre-defined function `text_to_ids` that converts the text
            # to a list of token IDs. Tokenization is essential for converting text to a format that the model can understand.
            tokenized = text_to_ids(text)
            total_tokens = len(tokenized)

            # Create input & target sequences from the tokenized data.
            # Slide over the tokenized text with a window of size `max_seq_length`.
            # The input sequence starts from position `i` to `i + max_seq_length`.
            # The target sequence starts from `i + 1` to `i + max_seq_length + 1`, effectively shifting the input by one token.
            for i in range(0, total_tokens - max_seq_length):
                input_seq = tokenized[i : i + max_seq_length]
                target_seq = tokenized[i + 1 : i + max_seq_length + 1]
                # Append the (input_seq, target_seq) pair to the sequences list.
                self.sequences.append((input_seq, target_seq))

    def __len__(self) -> int:
        """
        Return the number of sequences in the dataset.

        Returns:
        - int: The total number of (input, target) pairs in the dataset.
        """
        # The length of the dataset corresponds to the number of input-target pairs created.
        return len(self.sequences)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Retrieve the input & target sequence at the specified index.

        Args:
        - idx (int): The index of the sequence to retrieve.

        Returns:
        - Tuple[torch.Tensor, torch.Tensor]: The input & target sequences as tensors of dtype `torch.long`.
        """
        # Fetch the input & target sequences from the list at the given index.
        input_seq, target_seq = self.sequences[idx]

        # Convert the input & target sequences to PyTorch tensors of dtype `torch.long`.
        # Tensors are used as input to PyTorch models for training.
        input_seq = torch.tensor(input_seq, dtype=torch.long)
        target_seq = torch.tensor(target_seq, dtype=torch.long)

        # Return the input & target sequence pair.
        return input_seq, target_seq

### **6.4. Main Run**

In [ ]:
#@title Hyperparameters

# Maximum sequence length: This is the number of tokens in each input sequence that the model will process.
# Longer sequences provide more context but require more computational power & memory.
max_seq_length = 128

# Proportion of the dataset used for validation: This specifies what fraction of the dataset will be reserved
# for validation to monitor the model's performance during training.
validation_rate = 0.05

# Batch size for training: The number of sequences processed in parallel during each training step.
# Larger batch sizes can lead to faster training but may require more GPU memory.
batch_size = 128

# Maximum number of training epochs: The number of complete passes through the training dataset.
# Training for more epochs can improve performance but risks overfitting if too high.
epochs = 20

# Transformer model parameters:
# Number of encoder layers: The depth of the Transformer model. More layers allow the model to learn
# more complex patterns but increase computation time & risk of overfitting.
num_layers = 4

# Dimensionality of the model: The size of the hidden representations used throughout the Transformer.
# This affects the model's capacity & its ability to represent complex relationships.
d_model = 256

# Dimensionality of the feed-forward network: The size of the intermediate feed-forward network in
# each Transformer layer. This helps expand & contract representations.
dff = 128

# Number of attention heads: The number of parallel attention mechanisms within each multi-head attention layer.
# More heads allow the model to focus on different parts of the input sequence simultaneously.
num_heads = 4

# Dropout rate for regularization: Helps prevent overfitting by randomly dropping units during training.
dropout_rate = 0.2

# Regularization:
# Weight decay for optimizer: A technique to prevent the model from overfitting by adding a penalty to large weights.
# This helps in improving generalization.
weight_decay = 1e-4

# Learning rate scheduler parameters:
# Number of warm-up steps: The initial period during training where the learning rate gradually increases.
# This helps stabilize training in the early stages & prevents vanishing gradients.
warmup_steps = 4000

In [ ]:
#@title Directory Management

# Construct a unique experiment name using various hyperparameters & a timestamp.
# This helps in keeping track of different experiments & their settings.
experiment_name = f"me_{max_seq_length:05d}_{d_model:05d}_{num_layers:05d}_{num_heads:05d}_{dff:05d}_{int(time.time())}"

# Define the parent directory where all data & experiment outputs will be stored.
parent_dir = '/content/drive/MyDrive/data_535743/nlp/data_me'

# Specify the directory containing your text files, which are used as the input data for training.
text_files_dir = os.path.join(parent_dir, 'texts')

# Create a directory specific to this experiment to store various outputs such as models & logs.
experiment_dir = os.path.join(parent_dir, experiment_name)

# Directory to save the tokenizer model, which converts text to token IDs & vice versa.
tokenizer_model_dir = os.path.join(experiment_dir, 'tokenizer/')

# Directory to store checkpoints during training. Checkpoints save model states periodically
# for later resumption or evaluation.
checkpoints_dir = os.path.join(experiment_dir, 'checkpoints/')

# Directory to store training logs, which include metrics & other relevant data for monitoring training.
logs_dir = os.path.join(experiment_dir, 'logs/')

# Create the directories if they do not already exist.
# This ensures that the required folder structure is set up before the training begins.
os.makedirs(experiment_dir, exist_ok=True)
os.makedirs(tokenizer_model_dir, exist_ok=True)
os.makedirs(checkpoints_dir, exist_ok=True)
os.makedirs(logs_dir, exist_ok=True)

In [ ]:
#@title Data Management & Retreival

# Collect all text data from the specified directory.
# This involves reading each .txt file to aggregate their content into a single text variable.
all_text = ''

# Create a list of full file paths for all .txt files in the `text_files_dir`.
text_files = [os.path.join(text_files_dir, f) for f in os.listdir(text_files_dir) if f.endswith('.txt')]

# Iterate through each file path to read & append the contents.
for file_path in text_files:
    # Open the file in read mode with UTF-8 encoding to handle various text characters.
    with open(file_path, 'r', encoding='utf-8') as f:
        # Append the content of each file to `all_text`, separated by a newline for readability.
        all_text += f.read() + '\n'

# Save the combined text data to a temporary file for training the SentencePiece tokenizer.
# This file will be used as input for the tokenizer training process.
temp_text_file = os.path.join(tokenizer_model_dir, 'all_text.txt')
with open(temp_text_file, 'w', encoding='utf-8') as f:
    f.write(all_text)

# Train a SentencePiece tokenizer on the combined text data.
# SentencePiece is a popular subword tokenization method used for preparing text for NLP tasks.
# `--model_prefix` specifies the output model prefix, `--vocab_size` sets the vocabulary size,
# & `--model_type=bpe` selects the Byte-Pair Encoding model type.
spm.SentencePieceTrainer.Train(
    f'--input={temp_text_file} --model_prefix={tokenizer_model_dir}/tokenizer --vocab_size=32000 '
    f'--character_coverage=0.9995 --model_type=bpe'
)

# Load the trained SentencePiece tokenizer to use for encoding & decoding text.
sp = spm.SentencePieceProcessor()
sp.Load(f"{tokenizer_model_dir}/tokenizer.model")

# Get & print the vocabulary size of the tokenizer, which is the number of unique tokens it can produce.
vocab_size = sp.GetPieceSize()
print(f"Vocabulary Size: {vocab_size}")

# Split the collected text files into training & validation sets based on `validation_rate`.
# This helps evaluate the model's performance on unseen data during training.
random.shuffle(text_files)
split_index = int((1 - validation_rate) * len(text_files))
train_files = text_files[:split_index]  # Training set files.
val_files = text_files[split_index:]    # Validation set files.

# Load the text data for training by reading each file & appending its content to `train_texts`.
train_texts = []
for file_path in train_files:
    with open(file_path, 'r', encoding='utf-8') as f:
        train_texts.append(f.read())

# Load the text data for validation by reading each file & appending its content to `val_texts`.
val_texts = []
for file_path in val_files:
    with open(file_path, 'r', encoding='utf-8') as f:
        val_texts.append(f.read())

# Create the training dataset using the `TextDataset` class, which tokenizes & splits the text
# into input-target pairs of the specified `max_seq_length`.
train_dataset = TextDataset(train_texts, max_seq_length)

# Create the validation dataset in the same manner as the training dataset.
val_dataset = TextDataset(val_texts, max_seq_length)

# Create data loaders for training & validation. These data loaders batch the data & shuffle it
# (for training only) to ensure that batches are processed in random order during training.
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

# Clear the output to keep the notebook interface clean after loading & preparing data.
clear_output()

# Print the number of examples in the training & validation datasets.
print(f"Number of training examples: {len(train_dataset)}")
print(f"Number of validation examples: {len(val_dataset)}")

In [ ]:
#@title Model Development
# Set the device for computation.
# If a CUDA-compatible GPU is available, it will be used; otherwise, the CPU will be utilized.
# This ensures that the model runs efficiently on the available hardware.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Instantiate the Transformer-based language model using the specified hyperparameters.
# The model is initialized with the vocabulary size, embedding dimension (d_model),
# number of attention heads, number of encoder layers, feed-forward network size (dff),
# & dropout rate for regularization. The model is then moved to the specified device (GPU or CPU).
model = TransformerLanguageModel(
    vocab_size=vocab_size, d_model=d_model, num_heads=num_heads,
    num_layers=num_layers, dff=dff, dropout_rate=dropout_rate
).to(device)

# Define the loss function for training.
# CrossEntropyLoss is used as it is well-suited for multi-class classification problems,
# which aligns with language modeling tasks where each token is predicted as one of many possible vocabulary items.
criterion = nn.CrossEntropyLoss()

# Define the optimizer for training the model.
# AdamW (a variant of Adam with decoupled weight decay) is chosen for its efficiency & performance
# in training deep learning models. It helps to prevent overfitting by applying L2 regularization (weight decay).
optimizer = optim.AdamW(model.parameters(), lr=1, weight_decay=weight_decay)

# Define the learning rate scheduler.
# LambdaLR is used to adjust the learning rate according to a custom schedule defined by the `lr_lambda` function.
# This helps in dynamically changing the learning rate during training, aiding convergence & stability.
scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)

# Clear the Jupyter Notebook's cell output to keep the interface clean.
clear_output()

# Print the total number of trainable parameters in the model.
# This is important for understanding the model's complexity & capacity.
# The parameter count gives insight into the model's potential for learning.
print(f"Total Parameters: {sum([np.prod(p.size()) for p in model.parameters()])}")


In [ ]:
#@title Model Training

# Initialize the best validation loss to a very high value to ensure the first model saved has the lowest validation loss.
best_val_loss = float('inf')

# Loop over the specified number of epochs to train the model.
for epoch in range(1, epochs + 1):
    # Print the current epoch number for progress tracking.
    print(f"Epoch {epoch}/{epochs}")
    # Record the start time of the epoch to calculate the time taken for training & validation.
    start_time = time.time()

    # Call the `train_epoch` function to train the model for one epoch & get training loss & accuracy.
    train_loss, train_accuracy = train_epoch(model, optimizer, scheduler, train_loader)
    # Call the `evaluate` function to evaluate the model on the validation set & get validation loss & accuracy.
    val_loss, val_accuracy = evaluate(model, val_loader)

    # Calculate the time taken for the epoch by finding the difference between the current time & `start_time`.
    elapsed = time.time() - start_time

    # Print the training & validation metrics for the current epoch, including the time taken.
    print(f"Epoch {epoch} | Time: {elapsed:.2f}s | Train Loss: {train_loss:.4f} | "
          f"Train Accuracy: {train_accuracy:.4f} | Val Loss: {val_loss:.4f} | "
          f"Val Accuracy: {val_accuracy:.4f}")

    # Check if the current validation loss is lower than the previously recorded best validation loss.
    if val_loss < best_val_loss:
        # Update the `best_val_loss` to the current validation loss.
        best_val_loss = val_loss
        # Save the model's state dictionary to the checkpoints directory, naming it according to the validation loss.
        torch.save(model.state_dict(), os.path.join(checkpoints_dir, f'best_model_on_vl_{int(time.time())}.pt'))
        print("Model saved.")
    else:
        # Save the model's state dictionary when validation loss has not improved, with a different naming convention.
        torch.save(model.state_dict(), os.path.join(checkpoints_dir, f'best_model_on_tr_{int(time.time())}.pt'))
        print("No validation improvement.")

# Save the final model state to the checkpoints directory after the last epoch.
torch.save(model.state_dict(), os.path.join(checkpoints_dir, 'best_model.pt'))

### **6.5. Inference Functions**

In [ ]:
#@title Typing Effect
# Function to print a single word with a typing effect.
def typing_effect(word: str, max_width: int = 100, current_length: int = 0, delay: float = 0.01, add_space_before: bool = True) -> int:
    """
    Prints a single word with a typing effect, simulating real-time typing on the console.

    Parameters:
    - word (str): The word to be printed with the typing effect.
    - max_width (int): The maximum width of the line before wrapping to a new line.
    - current_length (int): The current length of the printed line, used to check if wrapping is needed.
    - delay (float): Time delay between printing each character to simulate typing. Default is 0.01 seconds.
    - add_space_before (bool): Flag to indicate whether to add a space before printing the word.
      This helps maintain proper word spacing.

    Returns:
    - int: The updated current length of the printed line after printing the word.
    """

    # Add a space before the word if `add_space_before` is True and it is not the start of a new line.
    if add_space_before and current_length > 0:
        # Check if adding a space would exceed the max line width; if so, move to a new line.
        if current_length + 1 > max_width:
            sys.stdout.write('\n')  # Print a newline character.
            current_length = 0  # Reset current length as we moved to a new line.
        else:
            sys.stdout.write(' ')  # Print a space for word separation.
            current_length += 1  # Increment current length to account for the space.

    # Check if the current word fits within the remaining space on the current line.
    # If not, wrap to the next line.
    if current_length + len(word) > max_width:
        sys.stdout.write('\n')  # Print a newline to start a new line.
        current_length = 0  # Reset current length for the new line.

    # Iterate through each character in the word to print it one at a time.
    for char in word:
        sys.stdout.write(char)  # Print the character.
        sys.stdout.flush()  # Ensure the character is printed immediately.
        time.sleep(delay)  # Add a delay to simulate typing.
        current_length += 1  # Increment current length for each printed character.

    # Return the updated current length of the printed line.
    return current_length

In [ ]:
#@title Punctuation Mark
def is_punctuation(text: str) -> bool:
    """
    Checks if the given text starts with a punctuation character.

    This function helps identify whether the first character of the provided string is a punctuation mark.
    This can be useful in NLP tasks for preprocessing text, such as when tokenizing sentences or handling
    punctuation separately from words.

    Parameters:
    - text (str): The text to check. This should be a string of any length.

    Returns:
    - bool: True if the text starts with a punctuation character, False otherwise.

    The `string.punctuation` constant in Python provides a string containing all standard punctuation characters.
    For more on string handling, refer to:
    https://docs.python.org/3/library/string.html#string.punctuation
    """
    # Check if the text is not empty & whether the first character of the text is in `string.punctuation`.
    # `string.punctuation` includes characters such as '.', ',', '!', '?', etc.
    return text and text[0] in string.punctuation

In [ ]:
#@title Filtering

def top_k_top_p_filtering(logits: torch.Tensor, top_k: int = 10, top_p: float = 0.90, filter_value: float = -float('Inf')) -> torch.Tensor:
    """
    Filter a distribution of logits using top-k and/or nucleus (top-p) filtering.

    This function ensures that only the most probable tokens are considered during sampling by limiting
    the token selection to a specified top-k number of logits or by cumulative probability up to top-p.
    This helps to reduce the risk of sampling unlikely tokens, which can improve the quality of generated text.

    Parameters:
    - logits (torch.Tensor): The input tensor containing logits (predictions) for each token, shape (vocab_size,).
    - top_k (int): The number of top logits to keep. Tokens with logits outside this range are filtered out.
                   If set to 0, top-k filtering is not applied.
    - top_p (float): The cumulative probability threshold for nucleus sampling. Tokens with cumulative probability
                     above this value are filtered out. If set to 0.0, top-p filtering is not applied.
    - filter_value (float): The value to assign to filtered logits, effectively removing them from consideration.

    Returns:
    - torch.Tensor: The filtered logits tensor with low-probability tokens set to `filter_value`.

    For more details on these sampling techniques, see:
    https://arxiv.org/abs/1904.09751 (Holtzman et al., 2019 - Nucleus Sampling)
    """
    # Ensure that the logits tensor is 1D, which represents a single distribution over tokens.
    assert logits.dim() == 1, "Logits should be a 1D tensor representing a token distribution."

    # Ensure `top_k` is within the valid range (not greater than the number of available logits).
    top_k = min(top_k, logits.size(-1))  # Safety check to prevent selecting more tokens than available.

    if top_k > 0:
        # Identify indices where logits are below the top-k threshold.
        # `torch.topk` returns the top-k values, & the threshold is the smallest of these values.
        indices_to_remove = logits < torch.topk(logits, top_k)[0][-1]
        # Set the logits of tokens outside the top-k range to `filter_value` to exclude them.
        logits[indices_to_remove] = filter_value

    if top_p > 0.0:
        # Sort logits in descending order to find cumulative probabilities.
        sorted_logits, sorted_indices = torch.sort(logits, descending=True)
        # Calculate the cumulative probabilities of the sorted logits.
        cumulative_probs = torch.softmax(sorted_logits, dim=-1).cumsum(dim=-1)

        # Identify tokens where the cumulative probability exceeds the top-p threshold.
        sorted_indices_to_remove = cumulative_probs > top_p
        # Shift the mask to include only the tokens after the first that exceeds the threshold.
        sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
        # Ensure the first token is not removed (it should remain the most probable one).
        sorted_indices_to_remove[..., 0] = False

        # Map the filtered indices back to the original order.
        indices_to_remove = sorted_indices[sorted_indices_to_remove]
        # Set the logits of these tokens to `filter_value` to exclude them from sampling.
        logits[indices_to_remove] = filter_value

    # Return the logits tensor with low-probability tokens filtered out.
    return logits

In [ ]:
#@title Generate Text
def generate_text(prompt: str, max_length: int = 200, temperature: float = 0.75, top_k: int = 5, top_p: float = 0.9) -> str:
    """
    Function to generate text from a prompt using a trained Transformer model.
    The function prints words as soon as they are generated, handling punctuation to avoid extra spaces.

    Parameters:
    - prompt (str): The initial text prompt to start generating text.
    - max_length (int): The maximum number of tokens to generate.
    - temperature (float): Controls the randomness of the predictions by scaling the logits.
                           Higher values (e.g., 1.5) result in more random outputs, while lower values (e.g., 0.5)
                           make the model more deterministic.
    - top_k (int): The number of top probable tokens to keep for sampling (0 means no top-k filtering).
    - top_p (float): The cumulative probability for nucleus sampling. Only tokens with cumulative probability
                     up to `top_p` are considered (0.0 means no nucleus sampling).

    Returns:
    - str: The generated text based on the input prompt.

    For more on text generation with Transformer models, see:
    https://huggingface.co/blog/how-to-generate
    """
    # Set the model to evaluation mode to disable dropout & other training-specific layers.
    model.eval();

    # Add a space between the seed prompt & the next generated tokens
    if not prompt.endswith(' '):
        prompt = prompt + ' '

    # Display the input prompt using the typing effect
    prompt_words = prompt.split(' ')
    current_length = 0  # Initialize the current line length counter
    is_first_word = True  # Flag to handle spacing for the first word

    for word in prompt_words:
        # Handle newlines within the prompt
        if '\n' in word:
            parts = word.split('\n')
            for i, part in enumerate(parts):
                if i > 0:
                    sys.stdout.write('\n')  # Start a new line
                    current_length = 0
                if part:
                    # For the first word, do not add space before
                    add_space_before = False if is_first_word else not is_punctuation(part)
                    current_length = typing_effect(part, max_width=100, current_length=current_length, delay=0.025, add_space_before=add_space_before)
                    is_first_word = False
        else:
            add_space_before = False if is_first_word else not is_punctuation(word)
            current_length = typing_effect(word, max_width=100, current_length=current_length, delay=0.025, add_space_before=add_space_before)
            is_first_word = False
    sys.stdout.flush()  # Ensure all prompt text is printed before generation starts


    # Convert the input prompt to token IDs using the `text_to_ids` function.
    generated = text_to_ids(prompt)
    # Convert the token IDs to a tensor & add a batch dimension. Move to the appropriate device.
    input_ids = torch.tensor(generated, dtype=torch.long).unsqueeze(0).to(device)  # (1, seq_len)


    # Disable gradient calculations during generation for efficiency.
    with torch.no_grad():
        # Convert initial input IDs to a list for appending new tokens during generation.
        output_ids = input_ids[0].tolist()
        # Convert the initial token IDs back to text.
        previous_text = ids_to_text(output_ids)

        # Loop to generate tokens up to the specified `max_length`.
        for _ in range(max_length):
            # Ensure input size does not exceed the model's maximum sequence length.
            if input_ids.size(1) > max_seq_length:
                input_ids = input_ids[:, -max_seq_length:]

            # Generate a mask for the input sequence to prevent looking at future tokens.
            src_mask = generate_square_subsequent_mask(input_ids.size(1))

            # Forward pass through the model to get predictions.
            outputs = model(input_ids, src_mask=src_mask)  # (1, seq_len, vocab_size)

            # Extract the logits for the last token & adjust by the temperature for sampling variability.
            next_token_logits = outputs[0, -1, :] / temperature

            # Apply top-k & top-p filtering to the logits for sampling.
            filtered_logits = top_k_top_p_filtering(next_token_logits, top_k=top_k, top_p=top_p)

            # Convert logits to probabilities & sample the next token.
            probabilities = torch.softmax(filtered_logits, dim=-1)
            next_token = torch.multinomial(probabilities, num_samples=1)

            # Decode the newly generated token to text.
            new_token_id = next_token.item()
            new_text = ids_to_text(new_token_id).strip()

            # Determine whether to add a space before printing the new token, checking if it's punctuation.
            add_space_before = not is_punctuation(new_text)

            # Print the new token using the typing effect function.
            if new_text:
                current_length = typing_effect(new_text, max_width=100, current_length=current_length, delay=0.025, add_space_before=add_space_before)

            # Append the new token to `input_ids` for the next iteration.
            input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=1)
            output_ids = input_ids[0].tolist()

            # Convert the current sequence of token IDs to text for final output.
            current_text = ids_to_text(output_ids)

            # Stop generation if the model generates an end-of-sequence (EOS) token.
            if next_token.item() == sp.eos_id():
                break

    # Return the full generated text.
    generated_text = current_text
    return generated_text

### **6.6. Inference Main Run**

In [ ]:
#@title Load the Best Model Weights

# Define the experiment name, which encodes the hyperparameters & a timestamp.
# This is useful for organizing different training runs & results.
experiment_name = "me_00512_00512_00016_00004_00512_1730516504"

# Extract the hyperparameters from the experiment name by splitting the string & converting values to integers.
# This allows for easy reconstruction of the model's configuration.
max_seq_length, d_model, num_layers, num_heads, dff, timestamp = (int(val) for val in experiment_name.split('_')[1:])

# Set the dropout rate for the model, used for regularization to prevent overfitting.
dropout_rate = 0.2

# Set the weight decay for the optimizer to control regularization strength, helping prevent overfitting.
weight_decay = 1e-4

# Define the number of warmup steps for the learning rate scheduler.
# Warmup helps to gradually increase the learning rate at the start of training to improve convergence.
warmup_steps = 4000

# Define the parent directory where data & models are stored.
parent_dir = '/content/drive/MyDrive/data_535743/nlp/data_me'

# Define the directory containing the text files used for training or testing.
text_files_dir = os.path.join(parent_dir, 'texts')

# Construct paths to various directories related to the experiment for loading & saving data.
experiment_dir = os.path.join(parent_dir, experiment_name)
tokenizer_model_dir = os.path.join(experiment_dir, 'tokenizer/')
checkpoints_dir = os.path.join(experiment_dir, 'checkpoints/')
logs_dir = os.path.join(experiment_dir, 'logs/')

# Load the trained SentencePiece tokenizer model.
# SentencePiece helps convert text to token IDs & vice versa for input/output processing.
sp = spm.SentencePieceProcessor()
sp.Load(f"{tokenizer_model_dir}/tokenizer.model")

# Get the vocabulary size from the tokenizer, which is essential for defining the model's embedding layer.
vocab_size = sp.GetPieceSize()

# Set the device to CUDA if a GPU is available; otherwise, use the CPU.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialize the Transformer-based language model with the specified hyperparameters.
# The model is configured with the given vocabulary size, embedding dimension (d_model),
# number of attention heads, number of encoder layers, & feed-forward network size (dff).
model = TransformerLanguageModel(
    vocab_size=vocab_size,
    d_model=d_model,
    num_heads=num_heads,
    num_layers=num_layers,
    dff=dff,
    dropout_rate=dropout_rate
).to(device)

# Clear any previous outputs in the Jupyter Notebook to keep the interface clean.
clear_output()

# Print the total number of parameters in the model to assess its complexity & capacity.
print(f"Total Parameters: {sum([np.prod(p.size()) for p in model.parameters()])}")

try:
    # Attempt to load the best model weights from a saved checkpoint named 'best_model_cpu.pt'.
    # The `map_location` argument ensures compatibility with the current device (CPU or GPU).
    model.load_state_dict(torch.load(os.path.join(checkpoints_dir, 'best_model_cpu.pt'), map_location=device))
except:
    # If the specific checkpoint does not exist, find the most recent model file saved during training.
    # Sort the model files based on their naming convention to load the latest checkpoint.
    model_files = sorted(glob.glob(os.path.join(checkpoints_dir, 'best_model_on_tr_*.pt')))
    # model_files = sorted(glob.glob(os.path.join(checkpoints_dir, 'best_model_on_vl_*.pt')))
    # Load the state dictionary of the last (most recent) model file found.
    model.load_state_dict(torch.load(os.path.join(checkpoints_dir, model_files[-1]), map_location=device))


In [ ]:
#@title Generate Text from A Seed Prompt
prompt = "Hypersonic Technologies includes the complex multidisciplinary challenges associated with sustained hypersonic flight, spanning speeds from Mach 5 to orbital velocities"
generated_text = generate_text(prompt, max_length=100, temperature=0.75, top_k=5)


# <font color="#418FDE" size="6.5" uppercase>**A: Transformers Essentials**</font>
----

In this lecture, you learned to:

* Explain the Attention Mechanism & its components in transformer models.
* Identify Large Language Models (LLMs) tasks, architectures, data structures, & major models.  
* Develop prototype multi-head multi-layer transformers from scratch for Mechanical Engineering text generation.

In the next lecture (lecture B), we will go over ready-to-use language & multi-modal models.




